# Mechanistic Probing Chain — PI > RI in Transformers

**8-step experiment chain.** Each step builds on previous results.

Steps 1-4: Model-level analysis (no head dependency)  
Step 5: Identify primacy heads (per-head causal knockout)  
Steps 6-8: Validate and characterize identified heads

**Run top-to-bottom.** Model loads ONCE. Results save to Drive.

**Models tested:**
- [ ] Qwen/Qwen2.5-0.5B-Instruct
- [ ] Qwen/Qwen2.5-1.5B-Instruct
- [ ] Qwen/Qwen2.5-3B-Instruct
- [ ] google/gemma-3-1b-it

In [2]:
# ═══════════════════════════════════════════════════════════════════════════
# INSTALL (Colab only)
# ═══════════════════════════════════════════════════════════════════════════
# Fix numpy/pandas binary incompatibility on Colab
# Must restart runtime after this cell runs for the first time
!pip install -q 'numpy<2.0' --force-reinstall
!pip install -q transformer_lens transformers accelerate scipy einops jaxtyping

import importlib, sys
# Check if restart is needed
try:
    import numpy as np
    if np.dtype('float64').itemsize != 8:
        raise ValueError('numpy mismatch')
    from transformer_lens import HookedTransformer
    print('All imports OK — no restart needed')
except (ValueError, ImportError) as e:
    print(f'ERROR: {e}')
    print('>>> RESTART RUNTIME NOW (Runtime → Restart runtime) then re-run this cell <<<')

# HuggingFace auth for gated models (Gemma)
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded from Colab secrets')
except Exception:
    if 'HF_TOKEN' not in os.environ:
        print('WARNING: HF_TOKEN not set — gated models will fail')

All imports OK — no restart needed


In [3]:
os.environ['HF_TOKEN'] = "REDACTED_HF_TOKEN"

In [4]:
# ═══════════════════════════════════════════════════════════════════════════
# CONFIG — Change these before running
# ═══════════════════════════════════════════════════════════════════════════

CONFIG = {
    'model_name': 'Qwen/Qwen3-4B',
    # Previous runs:
    # 'model_name': 'Qwen/Qwen2.5-0.5B-Instruct',
    # 'model_name': 'Qwen/Qwen2.5-1.5B-Instruct',
    # 'model_name': 'google/gemma-3-1b-it',

    # Trials per condition per experiment
    'recal_trials': 30,       # Step 1: recalibration
    'main_trials': 100,       # Steps 2-5, 7-8
    'validation_trials': 200, # Step 6: knockout validation
    'sweep_trials': 50,       # Step 6 V2: second-point sweep

    # Head selection from Step 5 → Steps 6-8
    # Takes all heads with Δld > this threshold (data-driven, not fixed N)
    'primacy_head_threshold_ld': 2.0,  # minimum causal effect in logits
    'primacy_head_max': 10,            # cap to avoid too many

    # TransformerLens context buffer
    'n_ctx': 2048,

    # GPU config
    'gpu_ids': [0],            # e.g. [0] for single GPU, [0,1] for 7B models

    # Paths — results saved per model per operating point
    'save_dir': '/content/results',
    'save_dir_drive': '/content/drive/MyDrive/mechanistic_probing_results',
}

MODEL_NAME = CONFIG['model_name']
print(f'Config loaded: {MODEL_NAME}')

Config loaded: Qwen/Qwen3-4B


In [5]:
# ═══════════════════════════════════════════════════════════════════════════
# SETUP — Mount Drive, imports, shared utilities
# ═══════════════════════════════════════════════════════════════════════════
import torch
import json
import time
import random
import numpy as np
from pathlib import Path

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
    print('Not on Colab — using local paths')

os.makedirs(CONFIG['save_dir'], exist_ok=True)
os.makedirs(CONFIG['save_dir_drive'], exist_ok=True)

# ── 46 categories (same as local core/dataset.py) ──
ORIGINAL_CATEGORIES = [
    'visual art', 'tools', 'landform', 'musical instrument', 'gemstone',
    'fabric', 'tree species', 'cheese variety', 'architectural style',
    'cloud formation', 'bird species', 'culinary herb', 'flower species',
    'wine variety', 'dance style', 'pasta shape', 'literary genre',
    'cooking method', 'mathematical concept', 'weather phenomenon',
    'ocean current', 'mineral type', 'coffee variety', 'telescope type',
    'martial art', 'sea creature', 'psychology term', 'chemical element',
    'dinosaur genus', 'programming language', 'ancient civilization',
    'bridge type', 'photography technique', 'boat type', 'cartoon character',
    'stadium name', 'surgical procedure', 'constellation', 'spice blend',
    'guitar type', 'hat style', 'painting medium', 'volcano name',
    'fruit variety', 'sword type', 'board game',
]

SYSTEM_PROMPT = 'Answer with ONLY the exact value. No explanation.'

# ── 2300 single-token English words (same as local core/single_token_values.py) ──
# Verified single-token (space-prefixed) in Qwen2.5-0.5B tokenizer.
# Filtered per-model at load time.
SINGLE_TOKEN_VALUES = [
    "above", "absorb", "accent", "accept", "access", "accord", "accuse", "ace", "acre", "across",
    "act", "adapt", "adjust", "admire", "adopt", "advent", "affirm", "afford", "age", "agenda",
    "agent", "aid", "aim", "air", "alarm", "ale", "alert", "alien", "align", "allied",
    "alloy", "ally", "almost", "altar", "always", "amazed", "amber", "amount", "ample", "anchor",
    "angel", "anger", "animal", "ankle", "annual", "answer", "ant", "anyway", "ape", "appeal",
    "appear", "apple", "arc", "arch", "arena", "ark", "arm", "armor", "army", "around",
    "array", "arrest", "arrive", "arrow", "art", "artist", "ash", "aside", "asleep", "assert",
    "assess", "assign", "assure", "atlas", "attach", "attack", "attend", "attic", "audio", "audit",
    "august", "aunt", "author", "autumn", "avenue", "awake", "awe", "axe", "babe", "back",
    "backup", "bacon", "badge", "bag", "bail", "bait", "bakery", "bald", "ball", "ballot",
    "bamboo", "ban", "banana", "band", "bang", "bank", "banker", "banner", "bar", "bare",
    "barely", "bark", "barley", "barn", "barrel", "base", "basic", "basin", "basket", "bat",
    "batch", "bath", "battle", "bay", "beach", "beacon", "bead", "beam", "bean", "bear",
    "beard", "beast", "beauty", "become", "bed", "bee", "before", "begin", "behalf", "behave",
    "behind", "behold", "belief", "bell", "belly", "belong", "below", "belt", "bench", "bend",
    "beside", "bet", "betray", "better", "beyond", "bid", "bike", "bill", "bin", "bird",
    "birth", "bit", "bite", "bitter", "black", "blade", "blame", "bland", "blank", "blanket",
    "blast", "blaze", "bleed", "blend", "bless", "blind", "blink", "bliss", "blob", "block",
    "blood", "bloom", "blot", "blow", "blown", "bluff", "blunt", "blur", "board", "boat",
    "body", "bog", "boiler", "bold", "bolster", "bolt", "bomb", "bond", "bone", "bonus",
    "book", "boom", "boot", "booth", "bore", "boss", "bother", "bottle", "bottom", "bounce",
    "bounty", "bout", "bow", "bowl", "box", "boy", "brain", "brake", "brand", "brass",
    "brave", "breach", "bread", "break", "breast", "breath", "breed", "breeze", "brick", "bride",
    "bridge", "brief", "bright", "bring", "brink", "broad", "broken", "bronze", "brow", "brown",
    "brush", "brute", "bud", "budget", "buffet", "bug", "bulk", "bull", "bullet", "bump",
    "bun", "bunch", "bundle", "bunk", "burden", "bureau", "burger", "burn", "burner", "burst",
    "bus", "bush", "bust", "butter", "button", "buyer", "buzz", "cab", "cabin", "cable",
    "cafe", "cage", "cake", "calf", "calm", "cam", "came", "camel", "camp", "can",
    "candle", "candy", "cane", "canopy", "canyon", "cap", "cape", "car", "carbon", "card",
    "cargo", "carpet", "carrot", "cart", "carved", "case", "cash", "cast", "castle", "cat",
    "cattle", "causal", "cave", "cedar", "cell", "cement", "center", "cereal", "chain", "chair",
    "chalk", "chance", "change", "chant", "chaos", "chapel", "charge", "charm", "chart", "chase",
    "cheap", "check", "cheek", "cheer", "cheese", "cherry", "chess", "chest", "child", "chill",
    "chin", "china", "chip", "choir", "chord", "chose", "chrome", "chunk", "cigar", "cipher",
    "circle", "circus", "cite", "city", "civic", "clad", "claim", "clam", "clamp", "clan",
    "clap", "clash", "class", "claw", "clay", "clean", "clear", "clerk", "cliff", "climb",
    "cling", "clip", "clock", "clone", "close", "closet", "cloth", "cloud", "clown", "club",
    "clue", "clutch", "coach", "coal", "coast", "coat", "cob", "cod", "coffee", "cog",
    "coil", "coin", "cold", "collar", "colony", "color", "column", "comb", "combat", "comedy",
    "commit", "common", "cone", "convex", "cook", "cool", "cooler", "cop", "cope", "copper",
    "coral", "cord", "core", "cork", "corn", "corner", "cosmic", "cost", "costly", "cotton",
    "couch", "count", "county", "coup", "couple", "coupon", "court", "cousin", "cover", "cow",
    "crab", "crack", "craft", "crane", "crash", "crate", "crazy", "cream", "create", "credit",
    "creek", "crest", "crew", "crime", "crisis", "crisp", "crop", "cross", "crow", "crowd",
    "crown", "crude", "cruise", "crush", "cry", "cub", "cube", "cult", "cup", "cure",
    "curl", "curve", "custom", "cut", "cycle", "cyst", "dairy", "dam", "damage", "dance",
    "dancer", "danger", "dare", "dark", "darn", "dart", "dash", "dawn", "day", "dead",
    "deaf", "deal", "dean", "dear", "debris", "decade", "decay", "decent", "deck", "decree",
    "deed", "deem", "deep", "deeper", "deer", "defeat", "degree", "delta", "demand", "demon",
    "den", "denied", "denim", "dense", "dent", "depart", "deploy", "depot", "derby", "desert",
    "design", "desk", "detail", "detect", "devil", "devote", "dew", "dial", "dialog", "diary",
    "dice", "diet", "differ", "dig", "digest", "digit", "dim", "dime", "dine", "dinner",
    "dip", "direct", "dirt", "disc", "dish", "disk", "divine", "dizzy", "dock", "dodge",
    "dog", "dome", "donate", "donor", "doom", "door", "dose", "dot", "double", "doubt",
    "dough", "dove", "down", "draft", "dragon", "drain", "drawer", "dream", "dress", "drift",
    "drill", "drink", "drip", "drive", "driven", "drone", "drop", "drove", "drown", "drum",
    "drunk", "dry", "dub", "duck", "due", "duel", "dug", "duke", "dull", "dump",
    "dun", "dung", "dunk", "duo", "duplex", "during", "dusk", "dust", "duty", "dwarf",
    "dwell", "dye", "dying", "eager", "eagle", "ear", "earn", "earth", "ease", "easier",
    "east", "easy", "eat", "edge", "egg", "ego", "eight", "elbow", "elder", "elk",
    "elm", "ember", "emerge", "empire", "empty", "enable", "end", "enemy", "energy", "engage",
    "engine", "enjoy", "enough", "enter", "enzyme", "epic", "equal", "era", "error", "escape",
    "essay", "estate", "evade", "eve", "event", "every", "evolve", "exact", "exceed", "excess",
    "excuse", "exempt", "exile", "exist", "expand", "expect", "expert", "export", "extend", "extent",
    "extra", "eye", "fabric", "face", "fact", "fade", "faint", "fair", "fairy", "faith",
    "fall", "false", "fame", "famine", "fan", "fancy", "fare", "farm", "farmer", "fast",
    "fat", "fate", "father", "faucet", "fault", "fax", "feast", "feat", "fee", "feed",
    "fellow", "female", "fence", "ferry", "fever", "fiber", "field", "fifth", "fifty", "fig",
    "fight", "figure", "file", "fill", "film", "filter", "fin", "final", "find", "fine",
    "finger", "fir", "fire", "firm", "fiscal", "fish", "fist", "fit", "five", "fix",
    "flag", "flame", "flank", "flap", "flare", "flash", "flask", "flat", "flavor", "flaw",
    "flea", "fled", "flee", "fleet", "flesh", "flight", "flip", "flock", "flood", "floor",
    "flop", "flora", "flour", "flow", "flower", "fluffy", "fluid", "flush", "flute", "fly",
    "focus", "foe", "fog", "fold", "folk", "fond", "font", "food", "fool", "foot",
    "force", "ford", "forest", "forge", "forget", "fork", "form", "formal", "former", "fort",
    "forum", "foster", "foul", "found", "four", "fox", "frame", "freak", "freeze", "fresh",
    "friday", "fringe", "frog", "frost", "froze", "frozen", "fruit", "fry", "fuel", "full",
    "fun", "fund", "fur", "fuse", "fuzz", "gag", "galaxy", "gallon", "gamble", "game",
    "gang", "gap", "garage", "garden", "garlic", "gas", "gate", "gather", "gauge", "gaze",
    "gear", "gem", "gender", "gentle", "ghost", "giant", "gift", "gin", "ginger", "gist",
    "glad", "glare", "glass", "glide", "glob", "global", "globe", "glory", "gloss", "glove",
    "glow", "glue", "goat", "god", "gold", "golden", "golf", "gone", "good", "goose",
    "gorge", "gossip", "gown", "grab", "grace", "grade", "grain", "grand", "grape", "grasp",
    "grass", "grave", "gravel", "gray", "green", "greet", "grid", "grief", "grill", "grim",
    "grin", "grind", "grip", "grit", "groom", "gross", "group", "grow", "guard", "guess",
    "guest", "guide", "guilt", "guitar", "gum", "gun", "gust", "gut", "gutter", "gym",
    "habit", "hack", "hail", "hair", "half", "hall", "halo", "halt", "ham", "hammer",
    "hand", "handle", "hang", "happen", "harbor", "hare", "harm", "harsh", "haste", "hat",
    "haunt", "haven", "hay", "hazard", "haze", "head", "heap", "hear", "heart", "heat",
    "heaven", "hedge", "heel", "heir", "held", "hello", "helm", "helper", "hen", "hence",
    "herb", "herbal", "herd", "hid", "hidden", "hide", "high", "hike", "hill", "hint",
    "hip", "hire", "hit", "hitch", "hive", "hobby", "hog", "hold", "hole", "holy",
    "home", "hone", "honest", "honey", "honor", "hood", "hoodie", "hook", "hoop", "hop",
    "hope", "horn", "horror", "horse", "hose", "host", "hostel", "hot", "hour", "house",
    "hover", "hub", "hue", "hug", "hull", "hum", "human", "humble", "humor", "hunger",
    "hurl", "hurry", "hurt", "hustle", "hut", "hybrid", "ice", "icon", "idle", "ill",
    "image", "imp", "impact", "import", "inch", "income", "indoor", "infant", "inform", "inject",
    "ink", "inn", "insect", "inside", "insist", "insult", "intact", "intend", "intern", "invade",
    "invent", "invest", "invite", "inward", "ion", "ire", "iris", "iron", "island", "itch",
    "itself", "ivory", "jab", "jacket", "jag", "jail", "jam", "jar", "jaw", "jazz",
    "jean", "jerk", "jest", "jet", "jewel", "jig", "job", "jog", "joint", "joke",
    "joy", "judge", "jug", "juice", "jump", "jungle", "junior", "jury", "just", "kale",
    "kayak", "keen", "keep", "keeper", "kernel", "kettle", "key", "kid", "kidney", "kin",
    "kind", "king", "kit", "kite", "kitten", "knack", "knee", "knew", "knife", "knight",
    "knit", "knob", "knock", "knot", "lab", "label", "lace", "lack", "lad", "ladder",
    "lag", "laid", "lake", "lamb", "lame", "lamp", "lance", "land", "lane", "lap",
    "laptop", "large", "laser", "lash", "latch", "late", "latter", "laugh", "launch", "lava",
    "lavish", "law", "lawn", "lay", "layer", "lazy", "leader", "leaf", "league", "leak",
    "lean", "leap", "leash", "least", "leave", "led", "ledge", "left", "leg", "lemon",
    "lend", "lens", "lent", "lessen", "letter", "level", "lever", "levy", "liar", "lid",
    "lie", "life", "lift", "light", "like", "likely", "limb", "lime", "limp", "line",
    "linen", "linger", "link", "lint", "lip", "liquid", "listen", "lit", "litter", "little",
    "live", "liver", "load", "loaf", "loan", "lobby", "local", "lock", "lodge", "loft",
    "log", "logic", "lone", "lonely", "loop", "loose", "loot", "lord", "lore", "lot",
    "lotion", "loud", "love", "loved", "lovely", "low", "lower", "loyal", "luck", "lug",
    "lumber", "lump", "lunch", "lure", "lush", "lust", "mad", "made", "magic", "magnet",
    "maid", "maiden", "mail", "major", "male", "mall", "malt", "man", "manage", "mane",
    "manner", "map", "maple", "marble", "march", "mare", "margin", "mark", "market", "marsh",
    "mash", "mask", "mass", "mast", "master", "mat", "match", "mate", "math", "matter",
    "mayor", "maze", "meal", "meat", "medal", "media", "meet", "meld", "melt", "memo",
    "men", "mend", "mental", "mercy", "merge", "merger", "merit", "mesh", "met", "metal",
    "meter", "metric", "mice", "middle", "midst", "mighty", "mild", "mile", "milk", "mill",
    "mime", "mind", "minded", "mine", "mingle", "minor", "mint", "minus", "mirror", "misery",
    "mist", "mixer", "mob", "mobile", "mock", "mod", "model", "modern", "modest", "moist",
    "mold", "mole", "molt", "moment", "money", "monk", "monkey", "month", "mood", "moon",
    "mop", "moral", "moss", "mostly", "moth", "mother", "motor", "mound", "mouse", "mouth",
    "move", "movie", "much", "mud", "muddy", "mug", "mural", "muscle", "muse", "museum",
    "music", "must", "muster", "mute", "muzzle", "myself", "myth", "nag", "nail", "nap",
    "narrow", "nasal", "nature", "naval", "navy", "nearby", "nearly", "neat", "neck", "needle",
    "neon", "nerve", "nest", "net", "never", "nickel", "niece", "night", "nil", "nine",
    "nip", "nit", "noble", "nod", "node", "noise", "noon", "norm", "normal", "north",
    "nose", "notch", "note", "notice", "noun", "novel", "novice", "nude", "number", "nun",
    "nurse", "nut", "nylon", "oak", "oasis", "oat", "oath", "obtain", "occupy", "ocean",
    "odd", "odds", "ode", "offend", "office", "oil", "olive", "online", "onset", "onto",
    "open", "opera", "oppose", "opt", "optic", "option", "orange", "orb", "orbit", "ore",
    "organ", "origin", "orphan", "our", "outer", "outfit", "output", "oval", "oven", "owl",
    "oxide", "oxygen", "pace", "pack", "pact", "pad", "paddle", "page", "paid", "paint",
    "pair", "pal", "palace", "pale", "palm", "pan", "pane", "panel", "panic", "pantry",
    "paper", "parcel", "pardon", "parent", "park", "part", "party", "pass", "past", "pasta",
    "paste", "pastor", "pastry", "patch", "path", "patron", "patter", "pause", "pave", "paw",
    "pawn", "pay", "pea", "peach", "peak", "pear", "pearl", "pedal", "peek", "peel",
    "peer", "peg", "pen", "pencil", "penny", "people", "pepper", "perch", "peril", "period",
    "perk", "permit", "person", "pest", "pet", "pew", "phase", "phone", "piano", "pick",
    "pie", "piece", "pier", "pig", "pile", "pillar", "pillow", "pilot", "pin", "pinch",
    "pine", "pink", "pint", "pipe", "pirate", "pit", "pitch", "pixel", "place", "plague",
    "plain", "plane", "planet", "plank", "plant", "plaque", "plate", "player", "plaza", "plead",
    "pledge", "plot", "plug", "plum", "plunge", "plus", "ply", "pocket", "pod", "poem",
    "poet", "poetry", "point", "poke", "polar", "pole", "police", "polish", "polite", "poll",
    "polo", "pomp", "pond", "ponder", "pony", "pool", "poor", "pop", "pope", "pore",
    "pork", "port", "portal", "pose", "post", "poster", "pot", "potato", "potion", "pouch",
    "pound", "pour", "power", "pray", "prayer", "prefer", "press", "prey", "price", "pride",
    "prime", "prince", "print", "prior", "prison", "prize", "probe", "prod", "prompt", "proof",
    "prop", "propel", "prose", "proud", "proven", "prune", "pry", "pub", "pulp", "pump",
    "pun", "punch", "pup", "pupil", "puppet", "pure", "purge", "purse", "pursue", "push",
    "puzzle", "quaint", "quarry", "quartz", "queen", "quest", "quick", "quiet", "quilt", "quiz",
    "quota", "quote", "rabbit", "race", "rack", "racket", "radar", "radio", "raft", "rag",
    "rage", "raid", "rail", "rain", "raise", "rake", "rally", "ram", "ramp", "ran",
    "ranch", "random", "range", "ranger", "rant", "rap", "rapid", "rare", "rarely", "rash",
    "rat", "rate", "rather", "rave", "raw", "ray", "razor", "reach", "react", "read",
    "real", "reap", "rear", "reason", "rebel", "recall", "recess", "recipe", "reckon", "record",
    "red", "reduce", "reef", "reel", "refine", "reform", "refuse", "regard", "region", "regret",
    "reign", "rein", "reject", "relate", "relax", "relay", "remain", "remedy", "remote", "rent",
    "rental", "repair", "repeat", "rescue", "resent", "reside", "resign", "resort", "retain", "retire",
    "return", "reveal", "review", "revolt", "reward", "rib", "ribbon", "rice", "rich", "rid",
    "ride", "rider", "ridge", "rig", "rigid", "rim", "ring", "riot", "rip", "ripple",
    "rise", "risen", "risk", "river", "road", "roam", "roar", "roast", "rob", "robe",
    "robot", "robust", "rock", "rocket", "rocky", "rod", "rode", "rogue", "role", "roll",
    "roller", "roof", "room", "root", "rope", "rot", "rotate", "rotten", "rouge", "round",
    "rout", "route", "rover", "row", "royal", "rub", "rubber", "rubble", "rug", "rugby",
    "ruin", "ruins", "rum", "rumor", "run", "runner", "rural", "rush", "rust", "rustic",
    "rusty", "rut", "sac", "sack", "sad", "saddle", "safari", "safe", "sag", "sage",
    "sail", "saint", "sake", "salad", "salmon", "salon", "salt", "salute", "sample", "sand",
    "sandy", "sane", "sap", "sat", "sauce", "savage", "save", "saw", "scale", "scan",
    "scar", "scarce", "scarf", "scene", "scenic", "scent", "scope", "score", "scout", "scrap",
    "sea", "seal", "seam", "season", "seat", "secure", "seed", "seldom", "senior", "series",
    "serve", "settle", "severe", "sewing", "shade", "shaft", "shake", "shame", "shape", "share",
    "shark", "sharp", "shave", "shear", "shed", "sheep", "sheer", "shelf", "shell", "shield",
    "shift", "shine", "ship", "shirt", "shock", "shop", "shore", "short", "shot", "shout",
    "shove", "show", "shut", "siege", "sigh", "sight", "signal", "silent", "silk", "silver",
    "simple", "sin", "since", "sing", "singer", "sink", "sip", "sir", "sit", "site",
    "sixth", "sixty", "size", "skate", "sketch", "ski", "skill", "skull", "sky", "slab",
    "slam", "slap", "slate", "slave", "sled", "sleep", "slew", "slice", "slid", "slide",
    "slim", "slit", "slogan", "slope", "slot", "slug", "slur", "small", "smart", "smell",
    "smile", "smith", "smoke", "smooth", "snack", "snake", "snap", "snatch", "snow", "soak",
    "soap", "sob", "sock", "socket", "sod", "soil", "solar", "sole", "solemn", "solid",
    "solve", "son", "song", "sonic", "soon", "sore", "sort", "soul", "sour", "south",
    "sow", "soy", "spa", "space", "span", "spar", "spare", "spark", "spawn", "speak",
    "spear", "spice", "spill", "spine", "spiral", "spit", "spoke", "sponge", "spoon", "sport",
    "spot", "spray", "spree", "sprint", "spur", "spy", "squad", "squid", "stab", "stable",
    "staff", "stag", "stage", "stain", "stair", "stake", "stale", "stalk", "stall", "stamp",
    "stance", "stand", "staple", "stare", "stark", "start", "state", "statue", "steady", "steam",
    "steel", "steep", "steer", "stem", "step", "stern", "stew", "stick", "stiff", "still",
    "sting", "stir", "stitch", "stock", "stolen", "stone", "stood", "stool", "stop", "storm",
    "story", "stout", "stove", "strain", "strand", "strap", "straw", "stray", "stream", "strict",
    "strike", "string", "strip", "stripe", "stroke", "stub", "stud", "stump", "stun", "sturdy",
    "style", "sub", "submit", "subtle", "sugar", "suit", "sum", "sun", "sunny", "super",
    "supper", "supply", "sure", "surely", "surf", "surge", "survey", "swamp", "swap", "swarm",
    "sway", "swear", "sweat", "sweep", "sweet", "swept", "swift", "swim", "swirl", "switch",
    "sword", "swore", "symbol", "system", "tab", "table", "tablet", "tack", "tact", "tactic",
    "tad", "tag", "tail", "tale", "talent", "tall", "tan", "tank", "tap", "tape",
    "tar", "target", "tart", "task", "tax", "tea", "team", "tear", "teeth", "temple",
    "tempo", "ten", "tenant", "tender", "tennis", "term", "text", "theft", "theme", "thick",
    "thief", "thing", "think", "third", "thirst", "three", "threw", "thrive", "throne", "throw",
    "thumb", "thwart", "tick", "ticket", "tide", "tidy", "tie", "tiger", "tight", "tile",
    "till", "tilt", "timber", "time", "timer", "tin", "tiny", "tip", "tire", "tissue",
    "toast", "toe", "toilet", "token", "tomb", "ton", "tone", "tongue", "took", "tool",
    "tooth", "top", "torch", "tore", "torn", "tot", "total", "touch", "tough", "tour",
    "tow", "toward", "towel", "tower", "town", "toxic", "toy", "trace", "track", "trade",
    "trail", "train", "trait", "trap", "trash", "travel", "tray", "treat", "treaty", "tree",
    "trek", "trend", "trial", "tribal", "tribe", "trick", "trim", "trio", "trip", "trophy",
    "trot", "trout", "truck", "truly", "trunk", "trust", "truth", "try", "tub", "tube",
    "tug", "tumor", "tuna", "tune", "tunnel", "turf", "turn", "turtle", "twelve", "twin",
    "twist", "two", "type", "ugly", "ultra", "uncle", "under", "undone", "unfair", "unfold",
    "union", "unique", "unit", "unite", "unity", "unrest", "update", "uphold", "upper", "upset",
    "urban", "urge", "urn", "useful", "usual", "utter", "vacuum", "vain", "vale", "valid",
    "valley", "value", "van", "vanish", "vapor", "vast", "vat", "vault", "veil", "vein",
    "velvet", "vent", "verb", "verbal", "verse", "vessel", "vest", "vet", "vie", "view",
    "viewer", "vigor", "vim", "vine", "vinyl", "violet", "virtue", "virus", "visa", "visit",
    "vivid", "vocal", "vodka", "voice", "void", "volt", "volume", "vote", "voter", "vow",
    "voyage", "wag", "wage", "wagon", "waist", "wait", "wake", "walk", "wall", "walnut",
    "wand", "wander", "war", "ward", "warm", "warmth", "warn", "warp", "wart", "wash",
    "waste", "watch", "water", "wave", "wax", "weak", "weapon", "wear", "weary", "web",
    "wed", "wedge", "weed", "week", "weekly", "weld", "well", "went", "west", "wet",
    "wheat", "wheel", "while", "whim", "whip", "whisk", "white", "whole", "wide", "widen",
    "widow", "width", "wife", "wig", "wild", "will", "wilt", "win", "wind", "window",
    "wine", "wing", "wipe", "wire", "wisdom", "wise", "wish", "wit", "witch", "within",
    "wolf", "womb", "wonder", "woo", "wood", "wooden", "wool", "word", "wore", "work",
    "worker", "worm", "worn", "worry", "worst", "worth", "wound", "wow", "wrap", "wreck",
    "wrist", "yacht", "yak", "yap", "yard", "yaw", "year", "yell", "yield", "yoga",
    "young", "youth", "zap", "zeal", "zen", "zero", "zinc", "zip", "zone", "zoo",
]
# 2300 words — identical to core/single_token_values.py

print(f'Categories: {len(ORIGINAL_CATEGORIES)}, Candidate values: {len(SINGLE_TOKEN_VALUES)}')

Device: cuda
GPU: Tesla T4
Memory: 15.6 GB
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Categories: 46, Candidate values: 2300


In [6]:
# ═══════════════════════════════════════════════════════════════════════════
# LOAD MODEL (TransformerLens — loaded ONCE, used by all steps)
# ═══════════════════════════════════════════════════════════════════════════
from transformer_lens import HookedTransformer

# ── GPU selection ──
gpu_ids = CONFIG.get('gpu_ids', [0])
os.environ['CUDA_VISIBLE_DEVICES'] = ','.join(str(g) for g in gpu_ids)
n_devices = len(gpu_ids)
print(f'Using GPUs: {gpu_ids} (n_devices={n_devices})')

print(f'Loading {MODEL_NAME}...')
model = HookedTransformer.from_pretrained(
    MODEL_NAME, device=DEVICE, n_ctx=CONFIG['n_ctx'], n_devices=n_devices,
    torch_dtype=torch.float16,
)
model.eval()
tokenizer = model.tokenizer
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

N_LAYERS = model.cfg.n_layers
N_HEADS = model.cfg.n_heads
D_MODEL = model.cfg.d_model
MODEL_SHORT = MODEL_NAME.split('/')[-1]

n_params = sum(p.numel() for p in model.parameters())
print(f'Loaded: {n_params:,} params, {N_LAYERS}L, {N_HEADS}H, d={D_MODEL}')

# ── Verify single-token values against THIS model's tokenizer ──
# Same logic as core/single_token_values.verify_single_token()
VALUE_POOL = []
VALUE_TO_TID = {}
for w in SINGLE_TOKEN_VALUES:
    tids = tokenizer.encode(f' {w}', add_special_tokens=False)
    if len(tids) == 1:
        VALUE_POOL.append(w)
        VALUE_TO_TID[w] = tids[0]

print(f'Single-token value pool: {len(VALUE_POOL)}/{len(SINGLE_TOKEN_VALUES)} valid for {MODEL_SHORT}')
if len(VALUE_POOL) < 100:
    print('WARNING: Pool too small. Some experiments may fail.')

Loading Qwen/Qwen3-4B...


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

: 

: 

: 

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SHARED UTILITIES — used by all steps
# ═══════════════════════════════════════════════════════════════════════════

def format_for_chat(prompt):
    if hasattr(tokenizer, 'apply_chat_template'):
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': prompt},
        ]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return prompt


def build_trial(num_keys, num_updates, condition, seed):
    """Build one trial with single-token values."""
    rng = random.Random(seed)
    categories = rng.sample(ORIGINAL_CATEGORIES, min(num_keys, len(ORIGINAL_CATEGORIES)))
    total_needed = num_keys * num_updates
    selected = rng.sample(VALUE_POOL, total_needed)
    values_per_cat = {}
    idx = 0
    for cat in categories:
        values_per_cat[cat] = selected[idx:idx + num_updates]
        idx += num_updates

    test_cat = categories[seed % num_keys]
    items = []
    for cat in categories:
        for val in values_per_cat[cat]:
            items.append({'category': cat, 'value': val})

    rng.shuffle(items)
    for _ in range(100):
        ok = all(items[i]['category'] != items[i-1]['category'] for i in range(1, len(items)))
        if ok:
            break
        rng.shuffle(items)

    stream = '\n'.join(f"{it['category']}: {it['value']}" for it in items)
    query_word = 'first' if condition == 'RI' else 'last'
    cat_values = [it['value'] for it in items if it['category'] == test_cat]
    expected = cat_values[0] if condition == 'RI' else cat_values[-1]

    prompt = (
        f'Read the following key-value stream. Each key gets updated multiple times.\n\n'
        f'{stream}\n\n'
        f'What was the {query_word} value of {test_cat}?'
    )
    return {
        'prompt': prompt, 'condition': condition, 'expected': expected,
        'initial_value': cat_values[0], 'final_value': cat_values[-1],
    }


def get_tids(value):
    """Get both space-prefixed and bare token IDs."""
    tid_sp = VALUE_TO_TID.get(value, -1)
    tid_bare = tokenizer.encode(value, add_special_tokens=False)[0]
    return list(set([t for t in [tid_sp, tid_bare] if t >= 0]))


def compute_logit_diff(logits_at_pos, correct_tids, incorrect_tids):
    """logit(correct) - logit(incorrect). Wang et al. 2022."""
    correct_logit = max(logits_at_pos[t].item() for t in correct_tids)
    incorrect_logit = max(logits_at_pos[t].item() for t in incorrect_tids)
    return correct_logit - incorrect_logit


def flush_memory():
    """Force GPU memory cleanup. Call between steps and inside loops."""
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        torch.mps.empty_cache()


def gpu_mem_report():
    """Print current GPU memory usage."""
    if torch.cuda.is_available():
        used = torch.cuda.memory_allocated() / 1e9
        cached = torch.cuda.memory_reserved() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'  GPU memory: {used:.1f}GB used, {cached:.1f}GB reserved, {total:.1f}GB total')


def get_result_dir(keys, updates):
    """Get result directory: {save_dir}/{model_short}/{keys}k_{updates}u/"""
    d = Path(CONFIG['save_dir']) / MODEL_SHORT / f'{keys}k_{updates}u'
    d.mkdir(parents=True, exist_ok=True)
    return d


def get_drive_dir(keys, updates):
    """Get Drive backup directory."""
    d = Path(CONFIG['save_dir_drive']) / MODEL_SHORT / f'{keys}k_{updates}u'
    d.mkdir(parents=True, exist_ok=True)
    return d


def save_step(step_name, data, keys=None, updates=None):
    """Save results to local + Drive with proper folder structure."""
    import shutil
    data['model'] = MODEL_NAME
    data['timestamp'] = time.strftime('%Y-%m-%d %H:%M:%S')

    if keys is None or updates is None:
        point_b = CHAIN_STATE['operating_points'].get('B')
        if point_b:
            keys, updates = point_b
        else:
            keys, updates = 0, 0

    local_dir = get_result_dir(keys, updates)
    local_path = local_dir / f'{step_name}.json'
    with open(local_path, 'w') as f:
        json.dump(data, f, indent=2)

    try:
        drive_dir = get_drive_dir(keys, updates)
        drive_path = drive_dir / f'{step_name}.json'
        shutil.copy(local_path, drive_path)
        print(f'  Saved: {local_path} + Drive')
    except Exception as e:
        print(f'  Saved: {local_path} (Drive failed: {e})')
    # Flush GPU memory after each step save
    flush_memory()
    gpu_mem_report()
    return str(local_path)


def select_primacy_heads(causal_effect, threshold_ld=None, max_heads=None):
    """Select primacy heads from knockout sweep results.

    Takes all heads with Δld > threshold. Shows all candidates,
    returns up to max_heads. Data-driven, not fixed N.
    """
    if threshold_ld is None:
        threshold_ld = CONFIG['primacy_head_threshold_ld']
    if max_heads is None:
        max_heads = CONFIG['primacy_head_max']

    n_layers, n_heads = causal_effect.shape
    flat = [(causal_effect[l, h], l, h) for l in range(n_layers) for h in range(n_heads)]
    flat.sort(reverse=True)

    above_threshold = [(e, l, h) for e, l, h in flat if e > threshold_ld]

    print(f'\n  Heads with Δld > {threshold_ld}:')
    if not above_threshold:
        print(f'    NONE — primacy is distributed (no single head dominates)')
        print(f'    Top 5 for reference:')
        for rank, (e, l, h) in enumerate(flat[:5]):
            print(f'      {rank+1}. L{l}H{h}: Δld={e:+.3f}')
        selected = [(int(l), int(h)) for _, l, h in flat[:3]]
        print(f'    Using top 3 as fallback: {selected}')
    else:
        for rank, (e, l, h) in enumerate(above_threshold):
            marker = ' ← selected' if rank < max_heads else ' (over cap)'
            print(f'    {rank+1}. L{l}H{h}: Δld={e:+.3f}{marker}')
        selected = [(int(l), int(h)) for _, l, h in above_threshold[:max_heads]]

    return selected


# Global state passed between steps
CHAIN_STATE = {
    'operating_points': {},  # Set by Step 1
    'primacy_heads': [],     # Set by Step 5
}

print('Utilities loaded.')
gpu_mem_report()

---
## Step 1: Single-Token Recalibration

Sweep keys × updates with single-token values to find operating points.

**Output:** Operating points A (both work), B (PI cracking), C (deep asymmetry), D (multi-key)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 1: Recalibration sweep
# ═══════════════════════════════════════════════════════════════════════════
RECAL_TRIALS = CONFIG['recal_trials']
keys_grid = [1, 2, 3, 5]
updates_grid = [1, 2, 3, 5, 10, 15, 20]

print(f'Step 1: Recalibration ({len(keys_grid)}×{len(updates_grid)} grid, {RECAL_TRIALS} trials)')
print('=' * 60)

recal_results = {}
t0 = time.time()

for keys in keys_grid:
    for updates in updates_grid:
        total_needed = keys * updates
        if total_needed > len(VALUE_POOL):
            continue

        cell = {}
        for condition in ['RI', 'PI']:
            correct = 0
            for t_idx in range(RECAL_TRIALS):
                seed = hash((condition, t_idx, updates, keys, 'recal')) % (2**31)
                trial = build_trial(keys, updates, condition, seed)
                formatted = format_for_chat(trial['prompt'])
                tokens = model.to_tokens(formatted)
                with torch.no_grad():
                    logits = model(tokens)
                pred_tid = logits[0, -1].argmax().item()
                pred = tokenizer.decode([pred_tid]).strip()
                if pred.lower() == trial['expected'].lower():
                    correct += 1
            cell[condition] = correct / RECAL_TRIALS

        ck = f'{keys}k_{updates}u'
        recal_results[ck] = cell
        gap = cell['RI'] - cell['PI']
        print(f'  {ck:>8}: RI={cell["RI"]:.0%}  PI={cell["PI"]:.0%}  gap={gap:+.0%}')

        flush_memory()

# ── Auto-select operating points ──
candidates = []
for ck, cell in recal_results.items():
    parts = ck.replace('k_', ' ').replace('u', '').split()
    k, u = int(parts[0]), int(parts[1])
    candidates.append((ck, k, u, cell['RI'], cell['PI'], cell['RI'] - cell['PI']))

# Point A: both high
a_cands = [c for c in candidates if c[3] >= 0.6 and c[4] >= 0.5]
point_a = max(a_cands, key=lambda x: x[3] + x[4]) if a_cands else None

# Point B: RI works, PI failing, max RI
b_cands = [c for c in candidates if c[3] >= 0.5 and c[4] < 0.4 and c[5] >= 0.15]
point_b = max(b_cands, key=lambda x: x[3]) if b_cands else None

# Point C: deep asymmetry
c_cands = [c for c in candidates if c[3] >= 0.4 and c[4] <= 0.15]
point_c = max(c_cands, key=lambda x: x[5]) if c_cands else None

# Point D: multi-key (keys >= 3)
d_cands = [c for c in candidates if c[1] >= 3 and c[3] >= 0.4 and c[4] < 0.4]
point_d = max(d_cands, key=lambda x: x[3]) if d_cands else None

points = {'A': point_a, 'B': point_b, 'C': point_c, 'D': point_d}
print(f'\n{"=" * 60}')
print('SELECTED OPERATING POINTS:')
for name, p in points.items():
    if p:
        CHAIN_STATE['operating_points'][name] = (p[1], p[2])
        print(f'  Point {name}: {p[0]} — RI={p[3]:.0%}, PI={p[4]:.0%}, gap={p[5]:+.0%}')
    else:
        print(f'  Point {name}: NOT FOUND')

print(f'\n  ({time.time()-t0:.0f}s total)')
save_step('step1_recalibration', {'grid': recal_results, 'operating_points': CHAIN_STATE['operating_points']})

---
## Step 2: Logit Lens

Project residual stream through unembedding at each layer.  
Track P(initial_value) and P(final_value) at the answer position.

**Expected:** P(init) dominates in RI (correct). P(init) also dominates in PI (wrong — this IS the primacy bias).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 2: Logit Lens at Point B
# ═══════════════════════════════════════════════════════════════════════════
TRIALS = CONFIG['main_trials']
point_b = CHAIN_STATE['operating_points'].get('B')
if not point_b:
    raise ValueError('No Point B found in Step 1. Check recalibration.')
KEYS, UPDATES = point_b

print(f'Step 2: Logit Lens at Point B ({KEYS}k, {UPDATES}u), {TRIALS} trials')
print('=' * 60)

# Per-layer accumulators
layer_p_init = {cond: np.zeros(N_LAYERS) for cond in ['RI', 'PI']}
layer_p_final = {cond: np.zeros(N_LAYERS) for cond in ['RI', 'PI']}
accuracy = {'RI': 0, 'PI': 0}
t0 = time.time()

for t_idx in range(TRIALS):
    for cond in ['RI', 'PI']:
        seed = hash((cond, t_idx, UPDATES, KEYS, 'step2')) % (2**31)
        trial = build_trial(KEYS, UPDATES, cond, seed)

        init_tids = get_tids(trial['initial_value'])
        final_tids = get_tids(trial['final_value'])

        formatted = format_for_chat(trial['prompt'])
        tokens = model.to_tokens(formatted)

        with torch.no_grad():
            logits, cache = model.run_with_cache(tokens, names_filter=lambda name: 'resid_post' in name)

        pred = tokenizer.decode([logits[0, -1].argmax().item()]).strip()
        if pred.lower() == trial['expected'].lower():
            accuracy[cond] += 1

        # Logit lens at each layer
        for layer in range(N_LAYERS):
            resid = cache['resid_post', layer][0, -1, :]
            layer_logits = resid @ model.W_U + model.b_U
            probs = torch.softmax(layer_logits, dim=-1)
            layer_p_init[cond][layer] += max(probs[t].item() for t in init_tids)
            layer_p_final[cond][layer] += max(probs[t].item() for t in final_tids)

        del cache
    if (t_idx + 1) % 25 == 0:
        print(f'  {t_idx+1}/{TRIALS} done ({time.time()-t0:.0f}s)')
    flush_memory()

# Average
for cond in ['RI', 'PI']:
    layer_p_init[cond] /= TRIALS
    layer_p_final[cond] /= TRIALS

print(f'\nAccuracy: RI={accuracy["RI"]}/{TRIALS}, PI={accuracy["PI"]}/{TRIALS}')
print(f'Last layer: RI P(init)={layer_p_init["RI"][-1]:.3f} P(final)={layer_p_final["RI"][-1]:.3f}')
print(f'           PI P(init)={layer_p_init["PI"][-1]:.3f} P(final)={layer_p_final["PI"][-1]:.3f}')

save_step('step2_logit_lens', {
    'config': {'keys': KEYS, 'updates': UPDATES, 'trials': TRIALS},
    'accuracy': {c: accuracy[c]/TRIALS for c in ['RI', 'PI']},
    'p_init_ri': layer_p_init['RI'].tolist(),
    'p_final_ri': layer_p_final['RI'].tolist(),
    'p_init_pi': layer_p_init['PI'].tolist(),
    'p_final_pi': layer_p_final['PI'].tolist(),
})
print(f'({time.time()-t0:.0f}s)')

---
## Step 3: Positional Gradient (Primacy Cliff)

For each value position v_0, v_1, ..., v_N, measure P(v_i) at the last layer.

**Expected:** RI: sharp cliff (P concentrated on v_0). PI: spread across late positions.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 3: Positional Gradient
# ═══════════════════════════════════════════════════════════════════════════
print(f'Step 3: Positional Gradient at Point B ({KEYS}k, {UPDATES}u), {TRIALS} trials')
print('=' * 60)

# Accumulate P(v_i) for each position index
pos_probs = {cond: np.zeros(UPDATES) for cond in ['RI', 'PI']}
pos_counts = {cond: 0 for cond in ['RI', 'PI']}
t0 = time.time()

for t_idx in range(TRIALS):
    for cond in ['RI', 'PI']:
        seed = hash((cond, t_idx, UPDATES, KEYS, 'step3')) % (2**31)
        trial = build_trial(KEYS, UPDATES, cond, seed)

        formatted = format_for_chat(trial['prompt'])
        tokens = model.to_tokens(formatted)
        token_ids = tokens[0].tolist()

        with torch.no_grad():
            logits = model(tokens)

        probs = torch.softmax(logits[0, -1], dim=-1)

        # Find position of each value in the stream order
        # trial is built with cat_values in stream order
        # We need to find each value's token and its probability
        rng = random.Random(seed)
        categories = rng.sample(ORIGINAL_CATEGORIES, min(KEYS, len(ORIGINAL_CATEGORIES)))
        test_cat = categories[seed % KEYS]
        # Rebuild to get ordered values
        trial2 = build_trial(KEYS, UPDATES, cond, seed)
        # Get all test-cat values in stream order from the prompt
        # Parse from prompt: lines containing test_cat
        lines = trial2['prompt'].split('\n')
        ordered_values = []
        for line in lines:
            if f'{test_cat}:' in line:
                val = line.split(':')[-1].strip()
                ordered_values.append(val)

        if len(ordered_values) == UPDATES:
            for vi, val in enumerate(ordered_values):
                tids = get_tids(val)
                if tids:
                    p = max(probs[t].item() for t in tids)
                    pos_probs[cond][vi] += p
            pos_counts[cond] += 1

    if (t_idx + 1) % 25 == 0:
        print(f'  {t_idx+1}/{TRIALS} done ({time.time()-t0:.0f}s)')
    flush_memory()

for cond in ['RI', 'PI']:
    if pos_counts[cond] > 0:
        pos_probs[cond] /= pos_counts[cond]

print(f'\nRI P(v_i): {" ".join(f"{p:.4f}" for p in pos_probs["RI"])}')
print(f'PI P(v_i): {" ".join(f"{p:.4f}" for p in pos_probs["PI"])}')

if len(pos_probs['RI']) >= 2 and pos_probs['RI'][0] > 0.001:
    ratio = pos_probs['RI'][1] / pos_probs['RI'][0]
    print(f'RI v1/v0 ratio: {ratio:.3f} (0=sharp cliff)')

save_step('step3_positional_gradient', {
    'config': {'keys': KEYS, 'updates': UPDATES, 'trials': TRIALS},
    'ri_probs': pos_probs['RI'].tolist(),
    'pi_probs': pos_probs['PI'].tolist(),
})
print(f'({time.time()-t0:.0f}s)')

---
## Step 4: Activation Patching

Clean run (1 value, no interference) vs corrupted run (N values).  
Patch clean residual stream into corrupted run at each layer.

**Expected:** Late-layer patching at answer position recovers PI ~100%. Early layers do nothing.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 4: Activation Patching (logit_diff metric)
# ═══════════════════════════════════════════════════════════════════════════
print(f'Step 4: Activation Patching at Point B ({KEYS}k, {UPDATES}u), {TRIALS} PI trials')
print('=' * 60)

# Only PI condition — that's where patching matters
recovery_per_layer = [[] for _ in range(N_LAYERS)]
t0 = time.time()

for t_idx in range(TRIALS):
    seed = hash(('PI', t_idx, UPDATES, KEYS, 'step4')) % (2**31)

    # Build matched clean/corrupted pair
    rng = random.Random(seed)
    cats = rng.sample(ORIGINAL_CATEGORIES, min(KEYS, len(ORIGINAL_CATEGORIES)))
    test_cat = cats[seed % KEYS]
    total = KEYS * UPDATES
    selected = rng.sample(VALUE_POOL, total)
    vpc = {}
    idx = 0
    for cat in cats:
        vpc[cat] = selected[idx:idx+UPDATES]
        idx += UPDATES
    final_val = vpc[test_cat][-1]
    init_val = vpc[test_cat][0]

    # Clean: 1 value per key
    clean_items = []
    for cat in cats:
        clean_items.append({'category': cat, 'value': vpc[cat][-1] if cat == test_cat else vpc[cat][0]})
    rng2 = random.Random(seed + 1000)
    rng2.shuffle(clean_items)
    clean_stream = '\n'.join(f"{it['category']}: {it['value']}" for it in clean_items)
    clean_prompt = f'Read the following key-value stream. Each key gets updated multiple times.\n\n{clean_stream}\n\nWhat was the last value of {test_cat}?'

    # Corrupted: all updates
    corr_items = []
    for cat in cats:
        for val in vpc[cat]:
            corr_items.append({'category': cat, 'value': val})
    rng3 = random.Random(seed + 2000)
    rng3.shuffle(corr_items)
    corr_stream = '\n'.join(f"{it['category']}: {it['value']}" for it in corr_items)
    corr_prompt = f'Read the following key-value stream. Each key gets updated multiple times.\n\n{corr_stream}\n\nWhat was the last value of {test_cat}?'

    correct_tids = get_tids(final_val)
    wrong_tids = get_tids(init_val)
    if not wrong_tids:
        wrong_tids = correct_tids

    # Run clean
    clean_tokens = model.to_tokens(format_for_chat(clean_prompt))
    with torch.no_grad():
        clean_logits, clean_cache = model.run_with_cache(clean_tokens, names_filter=lambda name: 'resid_post' in name)
    clean_ld = compute_logit_diff(clean_logits[0, -1], correct_tids, wrong_tids)

    # Run corrupted
    corr_tokens = model.to_tokens(format_for_chat(corr_prompt))
    with torch.no_grad():
        corr_logits = model(corr_tokens)
    corr_ld = compute_logit_diff(corr_logits[0, -1], correct_tids, wrong_tids)

    denom = clean_ld - corr_ld
    if abs(denom) < 0.01:
        del clean_cache
        continue

    # Patch each layer
    clean_ans = clean_tokens.shape[1] - 1
    corr_ans = corr_tokens.shape[1] - 1

    for layer in range(N_LAYERS):
        clean_resid = clean_cache['resid_post', layer][0, clean_ans, :].clone()
        def make_hook(act):
            def hook_fn(activation, hook):
                activation[0, corr_ans, :] = act
                return activation
            return hook_fn

        with torch.no_grad():
            patched_logits = model.run_with_hooks(
                corr_tokens, fwd_hooks=[(f'blocks.{layer}.hook_resid_post', make_hook(clean_resid))])
        patched_ld = compute_logit_diff(patched_logits[0, -1], correct_tids, wrong_tids)
        recovery = (patched_ld - corr_ld) / denom
        recovery_per_layer[layer].append(recovery)

    del clean_cache
    if (t_idx + 1) % 10 == 0:
        print(f'  {t_idx+1}/{TRIALS} done ({time.time()-t0:.0f}s)')
    flush_memory()

# Summarize
avg_recovery = [np.mean(r) if r else 0 for r in recovery_per_layer]
n_valid = [len(r) for r in recovery_per_layer]

print(f'\nPI recovery by layer:')
for l in range(0, N_LAYERS, max(1, N_LAYERS // 8)):
    bar = '█' * int(min(avg_recovery[l], 1.5) * 20) if avg_recovery[l] > 0 else ''
    print(f'  L{l:>2}: {avg_recovery[l]:>+7.0%} {bar}  (n={n_valid[l]})')

save_step('step4_activation_patching', {
    'config': {'keys': KEYS, 'updates': UPDATES, 'trials': TRIALS},
    'avg_recovery': avg_recovery,
    'n_valid': n_valid,
})
print(f'({time.time()-t0:.0f}s)')

---
## Step 5: Per-Head Causal Knockout ★

Zero out each head individually, measure PI logit_diff change.
**This is the key experiment** — identifies which heads cause primacy.

**Output:** Ranked list of primacy heads (positive Δld = helps PI when removed).  
Top N heads passed to Steps 6-8.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 5: Per-Head Knockout (25a)
# ═══════════════════════════════════════════════════════════════════════════
TRIALS = CONFIG['main_trials']
total_heads = N_LAYERS * N_HEADS

print(f'Step 5: Per-Head Knockout at Point B ({KEYS}k, {UPDATES}u)')
print(f'  {N_LAYERS} layers × {N_HEADS} heads = {total_heads} heads × {TRIALS} trials')
print('=' * 60)

# Build PI trials
trial_data = []
for t_idx in range(TRIALS):
    seed = hash(('PI', t_idx, UPDATES, KEYS, 'step5')) % (2**31)
    trial = build_trial(KEYS, UPDATES, 'PI', seed)
    formatted = format_for_chat(trial['prompt'])
    tokens = model.to_tokens(formatted)
    correct_tids = get_tids(trial['expected'])
    wrong_tids = get_tids(trial['initial_value'])
    if not wrong_tids:
        wrong_tids = correct_tids
    trial_data.append({'tokens': tokens, 'correct_tids': correct_tids, 'wrong_tids': wrong_tids})

# Baseline
print('  Running baseline...')
baseline_lds = []
baseline_correct = 0
for td in trial_data:
    with torch.no_grad():
        logits = model(td['tokens'])
    ld = compute_logit_diff(logits[0, -1], td['correct_tids'], td['wrong_tids'])
    baseline_lds.append(ld)
    if logits[0, -1].argmax().item() in td['correct_tids']:
        baseline_correct += 1
baseline_acc = baseline_correct / len(trial_data)
baseline_mean_ld = np.mean(baseline_lds)
print(f'  Baseline PI: {baseline_acc:.0%}, mean_ld={baseline_mean_ld:.2f}')

# Sweep all heads
t0 = time.time()
causal_effect = np.zeros((N_LAYERS, N_HEADS))
knockout_acc = np.zeros((N_LAYERS, N_HEADS))

for layer in range(N_LAYERS):
    for head in range(N_HEADS):
        def make_hook(h):
            def hook_fn(z, hook):
                z[:, :, h, :] = 0.0
                return z
            return hook_fn
        hook = (f'blocks.{layer}.attn.hook_z', make_hook(head))

        ko_deltas = []
        ko_correct = 0
        for i, td in enumerate(trial_data):
            with torch.no_grad():
                logits = model.run_with_hooks(td['tokens'], fwd_hooks=[hook])
            ld = compute_logit_diff(logits[0, -1], td['correct_tids'], td['wrong_tids'])
            ko_deltas.append(ld - baseline_lds[i])
            if logits[0, -1].argmax().item() in td['correct_tids']:
                ko_correct += 1

        causal_effect[layer, head] = np.mean(ko_deltas)
        knockout_acc[layer, head] = ko_correct / len(trial_data)

    elapsed = time.time() - t0
    done = (layer + 1) * N_HEADS
    eta = (elapsed / done) * (total_heads - done) if done > 0 else 0
    print(f'  Layer {layer}/{N_LAYERS-1} ({done}/{total_heads}, {elapsed:.0f}s, ETA {eta/60:.0f}min)')
    flush_memory()

# Select heads — data-driven, shows all above threshold
primacy_heads = select_primacy_heads(causal_effect)
CHAIN_STATE['primacy_heads'] = primacy_heads

# Also show top retrieval heads
flat = [(causal_effect[l, h], l, h) for l in range(N_LAYERS) for h in range(N_HEADS)]
flat.sort(reverse=True)

print(f'\nTop 5 retrieval heads (knockout hurts PI):')
for rank, (effect, l, h) in enumerate(flat[-5:][::-1]):
    print(f'  {rank+1}. L{l}H{h}: Δld={effect:+.3f}, KO acc={knockout_acc[l, h]:.0%}')

save_step('step5_per_head_knockout', {
    'config': {'keys': KEYS, 'updates': UPDATES, 'trials': TRIALS},
    'baseline': {'accuracy': baseline_acc, 'mean_ld': baseline_mean_ld},
    'causal_effect': causal_effect.tolist(),
    'knockout_accuracy': knockout_acc.tolist(),
    'selected_primacy_heads': [{'layer': l, 'head': h, 'delta_ld': float(causal_effect[l,h])} for l, h in primacy_heads],
    'top_primacy': [{'layer': int(l), 'head': int(h), 'delta_ld': float(e)} for e, l, h in flat[:20]],
    'top_retrieval': [{'layer': int(l), 'head': int(h), 'delta_ld': float(e)} for e, l, h in flat[-20:][::-1]],
    'selection_threshold_ld': CONFIG['primacy_head_threshold_ld'],
})
print(f'\n★ Primacy heads for Steps 6-8: {primacy_heads}')
print(f'({(time.time()-t0)/60:.1f} minutes)')

---
## Step 6: Knockout Validation

Three-way validation of Step 5 heads:
- **V1:** Knockout on BOTH RI and PI — primacy-specific or general loss?
- **V2:** Full sweep at Point C — same heads rank high?
- **V3:** 200-trial CIs for statistical confidence

**Uses:** `CHAIN_STATE['primacy_heads']` from Step 5

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 6: Knockout Validation
# ═══════════════════════════════════════════════════════════════════════════
VAL_TRIALS = CONFIG['validation_trials']
SWEEP_TRIALS = CONFIG['sweep_trials']
target_heads = CHAIN_STATE['primacy_heads']
point_c = CHAIN_STATE['operating_points'].get('C')

print(f'Step 6: Knockout Validation')
print(f'  Target heads: {target_heads}')
print(f'  V1: {VAL_TRIALS} trials × RI+PI at Point B')
if point_c:
    print(f'  V2: {SWEEP_TRIALS} trials × all heads at Point C ({point_c[0]}k,{point_c[1]}u)')
print('=' * 60)

# ── V1: RI + PI knockout ──
print(f'\nV1: RI + PI knockout at Point B...')
t0 = time.time()

v1_results = {}
for cond in ['RI', 'PI']:
    # Baseline
    bl_correct = 0
    bl_lds = []
    trials_v1 = []
    for t_idx in range(VAL_TRIALS):
        seed = hash((cond, t_idx, UPDATES, KEYS, 'step6v1')) % (2**31)
        trial = build_trial(KEYS, UPDATES, cond, seed)
        formatted = format_for_chat(trial['prompt'])
        tokens = model.to_tokens(formatted)
        correct_tids = get_tids(trial['expected'])
        wrong_val = trial['final_value'] if cond == 'RI' else trial['initial_value']
        wrong_tids = get_tids(wrong_val)
        if not wrong_tids:
            wrong_tids = correct_tids
        trials_v1.append({'tokens': tokens, 'correct_tids': correct_tids, 'wrong_tids': wrong_tids})

        with torch.no_grad():
            logits = model(tokens)
        ld = compute_logit_diff(logits[0, -1], correct_tids, wrong_tids)
        bl_lds.append(ld)
        if logits[0, -1].argmax().item() in correct_tids:
            bl_correct += 1

    bl_acc = bl_correct / VAL_TRIALS

    # Knockout each target head
    for layer, head in target_heads:
        def make_hook(h):
            def hook_fn(z, hook):
                z[:, :, h, :] = 0.0
                return z
            return hook_fn
        hook = (f'blocks.{layer}.attn.hook_z', make_hook(head))

        ko_correct = 0
        ko_deltas = []
        for i, td in enumerate(trials_v1):
            with torch.no_grad():
                logits = model.run_with_hooks(td['tokens'], fwd_hooks=[hook])
            ld = compute_logit_diff(logits[0, -1], td['correct_tids'], td['wrong_tids'])
            ko_deltas.append(ld - bl_lds[i])
            if logits[0, -1].argmax().item() in td['correct_tids']:
                ko_correct += 1

        ko_acc = ko_correct / VAL_TRIALS
        delta_ld = np.mean(ko_deltas)

        key = f'L{layer}H{head}_{cond}'
        v1_results[key] = {'baseline_acc': bl_acc, 'ko_acc': ko_acc, 'delta_ld': delta_ld}

        # Wilson CI
        z = 1.96
        n = VAL_TRIALS
        d = 1 + z**2/n
        center = (ko_acc + z**2/(2*n)) / d
        margin = z * np.sqrt((ko_acc*(1-ko_acc) + z**2/(4*n))/n) / d
        ci_lo, ci_hi = max(0, center-margin), min(1, center+margin)

        verdict = 'PRIMACY' if (cond == 'PI' and delta_ld > 1) or (cond == 'RI' and delta_ld < -1) else 'WEAK'
        print(f'  L{layer}H{head} {cond}: {bl_acc:.0%}→{ko_acc:.0%} [{ci_lo:.0%}-{ci_hi:.0%}] Δld={delta_ld:+.2f} {verdict}')

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ── V2: Sweep at Point C ──
v2_results = {}
if point_c:
    C_KEYS, C_UPDATES = point_c
    print(f'\nV2: Full sweep at Point C ({C_KEYS}k,{C_UPDATES}u), {SWEEP_TRIALS} trials...')

    # Build trials
    v2_trials = []
    v2_bl_lds = []
    for t_idx in range(SWEEP_TRIALS):
        seed = hash(('PI', t_idx, C_UPDATES, C_KEYS, 'step6v2')) % (2**31)
        trial = build_trial(C_KEYS, C_UPDATES, 'PI', seed)
        formatted = format_for_chat(trial['prompt'])
        tokens = model.to_tokens(formatted)
        correct_tids = get_tids(trial['expected'])
        wrong_tids = get_tids(trial['initial_value'])
        if not wrong_tids:
            wrong_tids = correct_tids
        v2_trials.append({'tokens': tokens, 'correct_tids': correct_tids, 'wrong_tids': wrong_tids})

        with torch.no_grad():
            logits = model(tokens)
        v2_bl_lds.append(compute_logit_diff(logits[0, -1], correct_tids, wrong_tids))

    # Sweep all heads
    v2_ce = np.zeros((N_LAYERS, N_HEADS))
    for layer in range(N_LAYERS):
        for head in range(N_HEADS):
            def make_hook(h):
                def hook_fn(z, hook):
                    z[:, :, h, :] = 0.0
                    return z
                return hook_fn
            hook = (f'blocks.{layer}.attn.hook_z', make_hook(head))
            deltas = []
            for i, td in enumerate(v2_trials):
                with torch.no_grad():
                    logits = model.run_with_hooks(td['tokens'], fwd_hooks=[hook])
                ld = compute_logit_diff(logits[0, -1], td['correct_tids'], td['wrong_tids'])
                deltas.append(ld - v2_bl_lds[i])
            v2_ce[layer, head] = np.mean(deltas)
        print(f'    Layer {layer}/{N_LAYERS-1}')
        flush_memory()

    v2_flat = sorted([(v2_ce[l,h],l,h) for l in range(N_LAYERS) for h in range(N_HEADS)], reverse=True)
    print(f'\n  Target heads at Point C:')
    for l, h in target_heads:
        rank = sum(1 for e,_,_ in v2_flat if e > v2_ce[l,h]) + 1
        print(f'    L{l}H{h}: rank={rank}/{total_heads}, Δld={v2_ce[l,h]:+.3f}')
    v2_results = {'causal_effect': v2_ce.tolist()}

save_step('step6_knockout_validation', {
    'config': {'keys': KEYS, 'updates': UPDATES, 'val_trials': VAL_TRIALS, 'sweep_trials': SWEEP_TRIALS},
    'target_heads': [{'layer': l, 'head': h} for l, h in target_heads],
    'v1': v1_results,
    'v2': v2_results,
})
print(f'\n({(time.time()-t0)/60:.1f} minutes)')

---
## Step 7: Instruction Sensitivity

Do the primacy heads attend to the query word "first"/"last"?

**Uses:** `CHAIN_STATE['primacy_heads']` from Step 5  
**Expected:** Heads ignore the query word entirely.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 7: Instruction Sensitivity (17b)
# ═══════════════════════════════════════════════════════════════════════════
TRIALS = CONFIG['main_trials']
target_heads = CHAIN_STATE['primacy_heads']

print(f'Step 7: Instruction Sensitivity for {target_heads}')
print('=' * 60)

# Accumulators
head_attn = {}
for l, h in target_heads:
    head_attn[(l,h)] = {c: {'query_word': [], 'initial': [], 'final': [], 'instruction': []}
                         for c in ['RI', 'PI']}
t0 = time.time()

for t_idx in range(TRIALS):
    seed = hash(('step7', t_idx, UPDATES, KEYS)) % (2**31)

    # Build matched RI/PI pair (identical except first/last)
    rng = random.Random(seed)
    cats = rng.sample(ORIGINAL_CATEGORIES, min(KEYS, len(ORIGINAL_CATEGORIES)))
    test_cat = cats[seed % KEYS]
    total = KEYS * UPDATES
    selected = rng.sample(VALUE_POOL, total)
    vpc = {}
    idx = 0
    for cat in cats:
        vpc[cat] = selected[idx:idx+UPDATES]
        idx += UPDATES

    items = []
    for cat in cats:
        for val in vpc[cat]:
            items.append({'category': cat, 'value': val})
    rng.shuffle(items)
    stream = '\n'.join(f"{it['category']}: {it['value']}" for it in items)

    init_tid = VALUE_TO_TID.get(vpc[test_cat][0], -1)
    final_tid = VALUE_TO_TID.get(vpc[test_cat][-1], -1)

    for cond in ['RI', 'PI']:
        qw = 'first' if cond == 'RI' else 'last'
        prompt = f'Read the following key-value stream. Each key gets updated multiple times.\n\n{stream}\n\nWhat was the {qw} value of {test_cat}?'
        formatted = format_for_chat(prompt)
        tokens = model.to_tokens(formatted)
        token_ids = tokens[0].tolist()
        str_tokens = model.to_str_tokens(formatted)

        with torch.no_grad():
            _, cache = model.run_with_cache(tokens, names_filter=lambda name: 'pattern' in name)

        init_pos = [i for i,t in enumerate(token_ids) if t == init_tid] if init_tid >= 0 else []
        final_pos = [i for i,t in enumerate(token_ids) if t == final_tid] if final_tid >= 0 else []

        # Find query word position
        qw_pos = []
        instr_pos = []
        in_query = False
        for i, st in enumerate(str_tokens):
            if 'What' in st:
                in_query = True
            if in_query and ('first' in st.lower() or 'last' in st.lower()):
                qw_pos.append(i)
            elif not in_query:
                instr_pos.append(i)

        for l, h in target_heads:
            pattern = cache['pattern', l]
            attn = pattern[0, h, -1, :]
            ha = head_attn[(l,h)][cond]
            ha['query_word'].append(attn[qw_pos].sum().item() if qw_pos else 0)
            ha['initial'].append(attn[init_pos].sum().item() if init_pos else 0)
            ha['final'].append(attn[final_pos].sum().item() if final_pos else 0)
            ha['instruction'].append(attn[instr_pos].sum().item() if instr_pos else 0)

        del cache
    if (t_idx+1) % 25 == 0:
        print(f'  {t_idx+1}/{TRIALS} done ({time.time()-t0:.0f}s)')
    flush_memory()

# Print
step7_data = []
for l, h in target_heads:
    print(f'\n  L{l}H{h}:')
    h_save = {'layer': l, 'head': h, 'regions': {}}
    for region in ['query_word', 'initial', 'final', 'instruction']:
        ri_avg = np.mean(head_attn[(l,h)]['RI'][region])
        pi_avg = np.mean(head_attn[(l,h)]['PI'][region])
        print(f'    {region:<15} RI={ri_avg:.4f}  PI={pi_avg:.4f}  diff={pi_avg-ri_avg:+.4f}')
        h_save['regions'][region] = {'RI': float(ri_avg), 'PI': float(pi_avg)}
    qw = np.mean(head_attn[(l,h)]['RI']['query_word'] + head_attn[(l,h)]['PI']['query_word'])
    print(f'    → {"IGNORES" if qw < 0.01 else "READS"} query word (avg attn={qw:.4f})')
    h_save['ignores_query'] = qw < 0.01
    step7_data.append(h_save)

save_step('step7_instruction_sensitivity', {
    'config': {'keys': KEYS, 'updates': UPDATES, 'trials': TRIALS},
    'heads': step7_data,
})
print(f'\n({time.time()-t0:.0f}s)')

---
## Step 8: Forced Attention

Force primacy heads to attend to the correct value position.  
Tests QK routing (A) vs indirect corruption (B).

**Uses:** Top 1 head from `CHAIN_STATE['primacy_heads']`  
**Expected:**  
- If forcing fixes PI → Mechanism A (QK routing problem)  
- If forcing doesn't fix PI → Mechanism B (indirect corruption)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 8: Forced Attention (18b) — top 1 head only
# ═══════════════════════════════════════════════════════════════════════════
TRIALS = CONFIG['main_trials']
force_heads = CHAIN_STATE['primacy_heads'][:1]  # Top 1 only for clean signal

print(f'Step 8: Forced Attention for {force_heads}')
print('=' * 60)

configs = ['baseline', 'knockout', 'force_correct', 'force_wrong']
results = {cfg: {c: {'correct': 0, 'total': 0, 'lds': []} for c in ['RI', 'PI']} for cfg in configs}
t0 = time.time()

for t_idx in range(TRIALS):
    for cond in ['RI', 'PI']:
        seed = hash((cond, t_idx, UPDATES, KEYS, 'step8')) % (2**31)
        trial = build_trial(KEYS, UPDATES, cond, seed)

        init_tid = VALUE_TO_TID.get(trial['initial_value'], -1)
        final_tid = VALUE_TO_TID.get(trial['final_value'], -1)
        formatted = format_for_chat(trial['prompt'])
        tokens = model.to_tokens(formatted)
        token_ids = tokens[0].tolist()

        init_pos = [i for i,t in enumerate(token_ids) if t == init_tid]
        final_pos = [i for i,t in enumerate(token_ids) if t == final_tid]
        if not init_pos or not final_pos:
            continue

        correct_pos = init_pos[0] if cond == 'RI' else final_pos[-1]
        wrong_pos = final_pos[-1] if cond == 'RI' else init_pos[0]

        init_tid_bare = tokenizer.encode(trial['initial_value'], add_special_tokens=False)[0]
        final_tid_bare = tokenizer.encode(trial['final_value'], add_special_tokens=False)[0]
        correct_tids = [init_tid_bare] if cond == 'RI' else [final_tid_bare]
        wrong_tids = [final_tid_bare] if cond == 'RI' else [init_tid_bare]

        for cfg in configs:
            hooks = []
            if cfg == 'knockout':
                for l, h in force_heads:
                    def make_ko(hh):
                        def hook_fn(z, hook): z[:,:,hh,:] = 0.0; return z
                        return hook_fn
                    hooks.append((f'blocks.{l}.attn.hook_z', make_ko(h)))
            elif cfg in ('force_correct', 'force_wrong'):
                tgt = correct_pos if cfg == 'force_correct' else wrong_pos
                for l, h in force_heads:
                    def make_force(hh, pos):
                        def hook_fn(pattern, hook):
                            pattern[0,hh,-1,:] = 0.0
                            if 0 <= pos < pattern.shape[-1]:
                                pattern[0,hh,-1,pos] = 1.0
                            return pattern
                        return hook_fn
                    hooks.append((f'blocks.{l}.attn.hook_pattern', make_force(h, tgt)))

            with torch.no_grad():
                logits = model.run_with_hooks(tokens, fwd_hooks=hooks) if hooks else model(tokens)

            pred = logits[0,-1].argmax().item()
            ld = compute_logit_diff(logits[0,-1], correct_tids, wrong_tids)
            results[cfg][cond]['correct'] += int(pred in correct_tids)
            results[cfg][cond]['total'] += 1
            results[cfg][cond]['lds'].append(ld)

    if (t_idx+1) % 25 == 0:
        print(f'  {t_idx+1}/{TRIALS} done ({time.time()-t0:.0f}s)')
    flush_memory()

# Print
print(f'\n{"Config":<20} {"RI":>6} {"PI":>6} {"Gap":>7} {"PI ld":>8}')
print('─' * 50)
for cfg in configs:
    ri = results[cfg]['RI']['correct'] / max(results[cfg]['RI']['total'], 1)
    pi = results[cfg]['PI']['correct'] / max(results[cfg]['PI']['total'], 1)
    pi_ld = np.mean(results[cfg]['PI']['lds']) if results[cfg]['PI']['lds'] else 0
    print(f'{cfg:<20} {ri:>5.0%} {pi:>5.0%} {ri-pi:>+6.0%} {pi_ld:>+7.2f}')

bl_pi = results['baseline']['PI']['correct'] / max(results['baseline']['PI']['total'], 1)
ko_pi = results['knockout']['PI']['correct'] / max(results['knockout']['PI']['total'], 1)
fc_pi = results['force_correct']['PI']['correct'] / max(results['force_correct']['PI']['total'], 1)

if fc_pi > bl_pi + 0.15 and fc_pi > ko_pi + 0.05:
    print(f'\n→ MECHANISM A: QK routing. Forcing fixes PI (+{fc_pi-bl_pi:.0%}).')
elif ko_pi > bl_pi + 0.15 and fc_pi <= ko_pi + 0.05:
    print(f'\n→ MECHANISM B: Indirect corruption. Knockout helps but forcing doesn\'t.')
elif ko_pi > bl_pi + 0.15 and fc_pi > ko_pi + 0.05:
    print(f'\n→ MECHANISM A+B: Mixed. Both routing and corruption.')
else:
    print(f'\n→ WEAK EFFECT at this operating point.')

save_step('step8_forced_attention', {
    'config': {'keys': KEYS, 'updates': UPDATES, 'trials': TRIALS},
    'heads': [{'layer': l, 'head': h} for l, h in force_heads],
    'results': {
        cfg: {c: {'accuracy': results[cfg][c]['correct']/max(results[cfg][c]['total'],1),
                   'mean_ld': float(np.mean(results[cfg][c]['lds'])) if results[cfg][c]['lds'] else 0,
                   'n': results[cfg][c]['total']}
              for c in ['RI','PI']}
        for cfg in configs},
})
print(f'\n({time.time()-t0:.0f}s)')

---
## Summary + Appendix Experiments

Steps 1-8 are the main chain. Below are supplementary experiments for the appendix.

| Appendix | What | Depends on |
|----------|------|-----------|
| A1 (exp 14) | PI mass distribution — where does peak P land? | Step 1 operating points |
| A2 (exp 19b) | Bias attention proof — blind vs oracle attention shift | Step 5 heads |
| A3 (exp 22) | Granular query patching — attn_out vs mlp_out vs resid | Step 1 operating points |
| A4 (exp 23) | Ablation + patching interaction — causal chain | Step 5 heads |
| A5 (exp 25c) | Observational metrics — DLA, entropy, copy score, primacy, sensitivity | Step 1 operating points |
| A6 (step 2) | Logit lens at Points A, C, D — interference progression | Step 1 operating points |

Note: Exp 19 (positional bias sweep) is a prerequisite context for A2 — it sweeps λ values to find the bias strength. A2 then shows WHERE the attention shifts under that bias. Both together tell the story: "blind bias fails because it goes to non-value tokens."

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# MAIN CHAIN SUMMARY
# ═══════════════════════════════════════════════════════════════════════════
import glob

print('=' * 60)
print(f'MECHANISTIC CHAIN COMPLETE: {MODEL_NAME}')
print('=' * 60)
print(f'\nModel: {MODEL_NAME} ({N_LAYERS}L, {N_HEADS}H, d={D_MODEL})')
print(f'Operating points: {CHAIN_STATE["operating_points"]}')
print(f'Primacy heads (from Step 5): {CHAIN_STATE["primacy_heads"]}')

print(f'\nSaved files:')
result_dir = Path(CONFIG['save_dir']) / MODEL_SHORT
for f in sorted(result_dir.rglob('*.json')):
    size = os.path.getsize(f)
    rel = f.relative_to(result_dir)
    print(f'  {rel} ({size:,} bytes)')

---
# Appendix Experiments

Optional supplementary experiments. Run after the main chain completes.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# A1: PI Mass Distribution (exp 14)
# Where does peak probability land in PI? At the actual last value, or somewhere else?
# Expected: PI peak drifts to ~80th percentile, not the actual last position.
# ═══════════════════════════════════════════════════════════════════════════
TRIALS = CONFIG['main_trials']
print(f'A1: PI Mass Distribution at Point B ({KEYS}k, {UPDATES}u), {TRIALS} trials')
print('=' * 60)

peak_positions = {'RI': [], 'PI': []}
t0 = time.time()

for t_idx in range(TRIALS):
    for cond in ['RI', 'PI']:
        seed = hash((cond, t_idx, UPDATES, KEYS, 'a1')) % (2**31)
        trial = build_trial(KEYS, UPDATES, cond, seed)
        formatted = format_for_chat(trial['prompt'])
        tokens = model.to_tokens(formatted)

        with torch.no_grad():
            logits = model(tokens)
        probs = torch.softmax(logits[0, -1], dim=-1)

        # Find which value position has max probability
        lines = trial['prompt'].split('\n')
        test_cat = None
        for line in lines:
            if 'What was the' in line:
                # Extract test category from query
                test_cat = line.split('value of ')[-1].rstrip('?')
                break

        ordered_values = []
        for line in lines:
            if test_cat and f'{test_cat}:' in line:
                val = line.split(':')[-1].strip()
                ordered_values.append(val)

        if len(ordered_values) == UPDATES:
            max_p = -1
            max_vi = 0
            for vi, val in enumerate(ordered_values):
                tids = get_tids(val)
                if tids:
                    p = max(probs[t].item() for t in tids)
                    if p > max_p:
                        max_p = p
                        max_vi = vi
            # Relative position (0=first, 1=last)
            rel_pos = max_vi / max(UPDATES - 1, 1)
            peak_positions[cond].append(rel_pos)

    if (t_idx + 1) % 25 == 0:
        print(f'  {t_idx+1}/{TRIALS} done ({time.time()-t0:.0f}s)')
    flush_memory()

for cond in ['RI', 'PI']:
    avg = np.mean(peak_positions[cond]) if peak_positions[cond] else 0
    at_first = sum(1 for p in peak_positions[cond] if p < 0.25) / max(len(peak_positions[cond]), 1)
    at_last = sum(1 for p in peak_positions[cond] if p > 0.75) / max(len(peak_positions[cond]), 1)
    print(f'\n{cond}: avg peak position = {avg:.2f} (0=first, 1=last)')
    print(f'  In first 25%: {at_first:.0%}   In last 25%: {at_last:.0%}')

save_step('appendix_a1_pi_mass', {
    'config': {'keys': KEYS, 'updates': UPDATES, 'trials': TRIALS},
    'ri_peak_positions': peak_positions['RI'],
    'pi_peak_positions': peak_positions['PI'],
})
print(f'({time.time()-t0:.0f}s)')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# A2: Bias Attention Proof (exp 19b)
# Where does attention shift under blind vs oracle positional bias?
# Uses primacy heads from Step 5.
# Expected: Blind bias → non-value tokens. Oracle → final value tokens.
# ═══════════════════════════════════════════════════════════════════════════
TRIALS = CONFIG['main_trials']
target_heads = CHAIN_STATE['primacy_heads']

if not target_heads:
    print('A2: SKIPPED — no primacy heads identified in Step 5')
else:
    print(f'A2: Bias Attention Proof for {target_heads}, {TRIALS} PI trials')
    print('=' * 60)

    LAMBDA = 10.0  # bias strength (from exp 19 results)
    modes = ['none', 'blind', 'oracle']

    # Accumulate attention by role
    attn_accum = {m: {'initial': 0, 'final': 0, 'other_value': 0, 'non_value': 0} for m in modes}
    n_trials_done = 0
    t0 = time.time()

    for t_idx in range(TRIALS):
        seed = hash(('PI', t_idx, UPDATES, KEYS, 'a2')) % (2**31)
        trial = build_trial(KEYS, UPDATES, 'PI', seed)

        init_tid = VALUE_TO_TID.get(trial['initial_value'], -1)
        final_tid = VALUE_TO_TID.get(trial['final_value'], -1)

        formatted = format_for_chat(trial['prompt'])
        tokens = model.to_tokens(formatted)
        token_ids = tokens[0].tolist()
        seq_len = tokens.shape[1]

        init_pos = set(i for i, t in enumerate(token_ids) if t == init_tid) if init_tid >= 0 else set()
        final_pos = set(i for i, t in enumerate(token_ids) if t == final_tid) if final_tid >= 0 else set()
        all_value_pos = init_pos | final_pos

        for mode in modes:
            hooks = []
            for layer, head in target_heads:
                if mode == 'blind':
                    def make_blind(h, lam=LAMBDA):
                        def hook_fn(pattern, hook):
                            seq = pattern.shape[-1]
                            ramp = torch.linspace(0, lam, seq, device=pattern.device)
                            pattern[0, h, -1, :] = pattern[0, h, -1, :] + ramp
                            # Re-normalize
                            pattern[0, h, -1, :] = torch.softmax(pattern[0, h, -1, :].log().clamp(min=-100) + ramp, dim=-1)
                            return pattern
                        return hook_fn
                    hooks.append((f'blocks.{layer}.attn.hook_pattern', make_blind(head)))
                elif mode == 'oracle':
                    def make_oracle(h, val_positions, lam=LAMBDA):
                        def hook_fn(pattern, hook):
                            seq = pattern.shape[-1]
                            bias = torch.zeros(seq, device=pattern.device)
                            for pos in val_positions:
                                if pos < seq:
                                    bias[pos] = lam * (pos / max(seq - 1, 1))
                            pattern[0, h, -1, :] = torch.softmax(pattern[0, h, -1, :].log().clamp(min=-100) + bias, dim=-1)
                            return pattern
                        return hook_fn
                    hooks.append((f'blocks.{layer}.attn.hook_pattern', make_oracle(head, list(all_value_pos))))

            with torch.no_grad():
                if hooks:
                    _, cache = model.run_with_cache(tokens, fwd_hooks=hooks)
                else:
                    _, cache = model.run_with_cache(tokens)

            # Extract attention for target heads
            for layer, head in target_heads:
                attn = cache['pattern', layer][0, head, -1, :]
                a_init = sum(attn[i].item() for i in init_pos)
                a_final = sum(attn[i].item() for i in final_pos)
                a_other = sum(attn[i].item() for i in range(seq_len) if i not in init_pos and i not in final_pos and i in all_value_pos)
                a_non = 1.0 - a_init - a_final - a_other
                attn_accum[mode]['initial'] += a_init
                attn_accum[mode]['final'] += a_final
                attn_accum[mode]['other_value'] += a_other
                attn_accum[mode]['non_value'] += a_non

            del cache
        n_trials_done += 1

        if (t_idx + 1) % 25 == 0:
            print(f'  {t_idx+1}/{TRIALS} done ({time.time()-t0:.0f}s)')
        flush_memory()

    # Normalize
    n = n_trials_done * len(target_heads)
    print(f'\n  {"Mode":<10} {"→Initial":>10} {"→Final":>10} {"→OtherVal":>10} {"→NonValue":>10}')
    print(f'  {"─" * 50}')
    for mode in modes:
        for role in attn_accum[mode]:
            attn_accum[mode][role] /= max(n, 1)
        a = attn_accum[mode]
        print(f'  {mode:<10} {a["initial"]:>10.3f} {a["final"]:>10.3f} {a["other_value"]:>10.3f} {a["non_value"]:>10.3f}')

    save_step('appendix_a2_bias_proof', {
        'config': {'keys': KEYS, 'updates': UPDATES, 'trials': TRIALS, 'lambda': LAMBDA},
        'heads': [{'layer': l, 'head': h} for l, h in target_heads],
        'attention_by_mode': attn_accum,
    })
    print(f'({time.time()-t0:.0f}s)')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# A3: Granular Query Patching (exp 22)
# Patch at query position: resid_post vs attn_out vs mlp_out, per layer.
# Expected: Only late-layer resid_post works. Neither attn nor mlp alone.
# ═══════════════════════════════════════════════════════════════════════════
TRIALS = CONFIG['main_trials']
print(f'A3: Granular Query Patching at Point B ({KEYS}k, {UPDATES}u), {TRIALS} PI trials')
print('=' * 60)

# Layers to test
step = max(1, N_LAYERS // 13)
test_layers = sorted(set(list(range(0, N_LAYERS, step)) + [N_LAYERS - 1]))
components = ['resid_post', 'attn_out', 'mlp_out']

recovery_data = {comp: {l: [] for l in test_layers} for comp in components}
t0 = time.time()

for t_idx in range(TRIALS):
    seed = hash(('PI', t_idx, UPDATES, KEYS, 'a3')) % (2**31)

    # Build matched pair
    rng = random.Random(seed)
    cats = rng.sample(ORIGINAL_CATEGORIES, min(KEYS, len(ORIGINAL_CATEGORIES)))
    test_cat = cats[seed % KEYS]
    total = KEYS * UPDATES
    selected = rng.sample(VALUE_POOL, total)
    vpc = {}
    idx = 0
    for cat in cats:
        vpc[cat] = selected[idx:idx+UPDATES]
        idx += UPDATES

    final_val = vpc[test_cat][-1]
    init_val = vpc[test_cat][0]
    correct_tids = get_tids(final_val)
    wrong_tids = get_tids(init_val)
    if not wrong_tids:
        wrong_tids = correct_tids

    # Clean
    clean_items = [{'category': cat, 'value': vpc[cat][-1] if cat == test_cat else vpc[cat][0]} for cat in cats]
    random.Random(seed+1000).shuffle(clean_items)
    clean_prompt = f'Read the following key-value stream. Each key gets updated multiple times.\n\n' + \
        '\n'.join(f"{it['category']}: {it['value']}" for it in clean_items) + f'\n\nWhat was the last value of {test_cat}?'

    # Corrupted
    corr_items = [{'category': cat, 'value': val} for cat in cats for val in vpc[cat]]
    random.Random(seed+2000).shuffle(corr_items)
    corr_prompt = f'Read the following key-value stream. Each key gets updated multiple times.\n\n' + \
        '\n'.join(f"{it['category']}: {it['value']}" for it in corr_items) + f'\n\nWhat was the last value of {test_cat}?'

    clean_tokens = model.to_tokens(format_for_chat(clean_prompt))
    corr_tokens = model.to_tokens(format_for_chat(corr_prompt))

    with torch.no_grad():
        clean_logits, clean_cache = model.run_with_cache(clean_tokens, names_filter=lambda name: 'resid_post' in name or 'attn_out' in name or 'mlp_out' in name)
    clean_ld = compute_logit_diff(clean_logits[0, -1], correct_tids, wrong_tids)

    with torch.no_grad():
        corr_logits = model(corr_tokens)
    corr_ld = compute_logit_diff(corr_logits[0, -1], correct_tids, wrong_tids)

    denom = clean_ld - corr_ld
    if abs(denom) < 0.01:
        del clean_cache
        continue

    # Find query positions
    str_tokens = model.to_str_tokens(format_for_chat(corr_prompt))
    clean_str = model.to_str_tokens(format_for_chat(clean_prompt))
    corr_qpos = []
    clean_qpos = []
    for i in range(len(str_tokens)-1, -1, -1):
        if 'What' in str_tokens[i]:
            corr_qpos = list(range(i, len(str_tokens)))
            break
    for i in range(len(clean_str)-1, -1, -1):
        if 'What' in clean_str[i]:
            clean_qpos = list(range(i, len(clean_str)))
            break

    n_patch = min(len(clean_qpos), len(corr_qpos))

    for layer in test_layers:
        for comp in components:
            if comp == 'resid_post':
                hook_name = f'blocks.{layer}.hook_resid_post'
                cache_key = ('resid_post', layer)
            elif comp == 'attn_out':
                hook_name = f'blocks.{layer}.hook_attn_out'
                cache_key = ('attn_out', layer)
            else:
                hook_name = f'blocks.{layer}.hook_mlp_out'
                cache_key = ('mlp_out', layer)

            clean_acts = [clean_cache[cache_key][0, clean_qpos[i], :].clone() for i in range(n_patch)]

            def make_hook(acts, positions):
                def hook_fn(activation, hook):
                    for i in range(len(acts)):
                        if positions[i] < activation.shape[1]:
                            activation[0, positions[i], :] = acts[i]
                    return activation
                return hook_fn

            with torch.no_grad():
                patched_logits = model.run_with_hooks(
                    corr_tokens, fwd_hooks=[(hook_name, make_hook(clean_acts, corr_qpos[:n_patch]))])
            patched_ld = compute_logit_diff(patched_logits[0, -1], correct_tids, wrong_tids)
            recovery = (patched_ld - corr_ld) / denom
            recovery_data[comp][layer].append(recovery)

    del clean_cache
    if (t_idx+1) % 10 == 0:
        print(f'  {t_idx+1}/{TRIALS} done ({time.time()-t0:.0f}s)')
    flush_memory()

print(f'\n  {"Layer":>6}  {"resid_post":>12}  {"attn_out":>12}  {"mlp_out":>12}')
print(f'  {"─" * 45}')
for layer in test_layers:
    vals = {c: np.mean(recovery_data[c][layer]) if recovery_data[c][layer] else 0 for c in components}
    print(f'  L{layer:>4}  {vals["resid_post"]:>+11.0%}  {vals["attn_out"]:>+11.0%}  {vals["mlp_out"]:>+11.0%}')

save_step('appendix_a3_query_patching', {
    'config': {'keys': KEYS, 'updates': UPDATES, 'trials': TRIALS},
    'test_layers': test_layers,
    'recovery': {comp: {str(l): float(np.mean(recovery_data[comp][l])) if recovery_data[comp][l] else 0
                        for l in test_layers} for comp in components},
})
print(f'({time.time()-t0:.0f}s)')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# A4: Ablation + Patching Interaction (exp 23)
# Does ablating primacy heads reduce the need for query-position patching?
# Uses primacy heads from Step 5.
# ═══════════════════════════════════════════════════════════════════════════
TRIALS = CONFIG['main_trials']
target_heads = CHAIN_STATE['primacy_heads']

if not target_heads:
    print('A4: SKIPPED — no primacy heads')
else:
    print(f'A4: Ablation + Patching for {target_heads}, {TRIALS} PI trials')
    print('=' * 60)

    step = max(1, N_LAYERS // 9)
    test_layers = sorted(set(list(range(0, N_LAYERS, step)) + [N_LAYERS - 1]))
    modes = ['normal', 'ablated']
    recovery = {m: {l: [] for l in test_layers} for m in modes}
    accuracy = {m: 0 for m in modes}
    t0 = time.time()

    for t_idx in range(TRIALS):
        seed = hash(('PI', t_idx, UPDATES, KEYS, 'a4')) % (2**31)

        rng = random.Random(seed)
        cats = rng.sample(ORIGINAL_CATEGORIES, min(KEYS, len(ORIGINAL_CATEGORIES)))
        test_cat = cats[seed % KEYS]
        selected = rng.sample(VALUE_POOL, KEYS * UPDATES)
        vpc = {}; idx = 0
        for cat in cats:
            vpc[cat] = selected[idx:idx+UPDATES]; idx += UPDATES

        final_val, init_val = vpc[test_cat][-1], vpc[test_cat][0]
        correct_tids, wrong_tids = get_tids(final_val), get_tids(init_val)
        if not wrong_tids: wrong_tids = correct_tids

        clean_items = [{'category': cat, 'value': vpc[cat][-1] if cat == test_cat else vpc[cat][0]} for cat in cats]
        random.Random(seed+1000).shuffle(clean_items)
        clean_prompt = f'Read the following key-value stream. Each key gets updated multiple times.\n\n' + \
            '\n'.join(f"{it['category']}: {it['value']}" for it in clean_items) + f'\n\nWhat was the last value of {test_cat}?'

        corr_items = [{'category': cat, 'value': val} for cat in cats for val in vpc[cat]]
        random.Random(seed+2000).shuffle(corr_items)
        corr_prompt = f'Read the following key-value stream. Each key gets updated multiple times.\n\n' + \
            '\n'.join(f"{it['category']}: {it['value']}" for it in corr_items) + f'\n\nWhat was the last value of {test_cat}?'

        clean_tokens = model.to_tokens(format_for_chat(clean_prompt))
        corr_tokens = model.to_tokens(format_for_chat(corr_prompt))

        # Find query positions
        clean_str = model.to_str_tokens(format_for_chat(clean_prompt))
        corr_str = model.to_str_tokens(format_for_chat(corr_prompt))
        clean_qpos = next((list(range(i, len(clean_str))) for i in range(len(clean_str)-1,-1,-1) if 'What' in clean_str[i]), [])
        corr_qpos = next((list(range(i, len(corr_str))) for i in range(len(corr_str)-1,-1,-1) if 'What' in corr_str[i]), [])
        n_patch = min(len(clean_qpos), len(corr_qpos))

        # Ablation hooks
        abl_hooks = []
        for l, h in target_heads:
            def make_ko(hh):
                def hook_fn(z, hook): z[:,:,hh,:] = 0.0; return z
                return hook_fn
            abl_hooks.append((f'blocks.{l}.attn.hook_z', make_ko(h)))

        for mode in modes:
            extra = abl_hooks if mode == 'ablated' else []

            # Clean with mode
            with torch.no_grad():
                if extra:
                    clean_logits = model.run_with_hooks(clean_tokens, fwd_hooks=extra)
                else:
                    clean_logits = model(clean_tokens)
            # Need cache for clean — run again with cache
            cache_hooks = []
            cache_dict = {}
            for layer in test_layers:
                def make_cache(l):
                    def hook_fn(act, hook):
                        cache_dict[('resid_post', l)] = act.detach()
                        return act
                    return hook_fn
                cache_hooks.append((f'blocks.{layer}.hook_resid_post', make_cache(layer)))
            with torch.no_grad():
                clean_logits = model.run_with_hooks(clean_tokens, fwd_hooks=extra + cache_hooks)
            clean_ld = compute_logit_diff(clean_logits[0, -1], correct_tids, wrong_tids)

            # Corrupted with mode
            with torch.no_grad():
                if extra:
                    corr_logits = model.run_with_hooks(corr_tokens, fwd_hooks=extra)
                else:
                    corr_logits = model(corr_tokens)
            corr_ld = compute_logit_diff(corr_logits[0, -1], correct_tids, wrong_tids)
            pred = corr_logits[0,-1].argmax().item()
            accuracy[mode] += int(pred in correct_tids)

            denom = clean_ld - corr_ld
            if abs(denom) < 0.01:
                continue

            # Patch at each test layer
            for layer in test_layers:
                if ('resid_post', layer) not in cache_dict:
                    continue
                clean_acts = [cache_dict[('resid_post', layer)][0, clean_qpos[i], :].clone() for i in range(n_patch)]
                def make_patch(acts, positions):
                    def hook_fn(activation, hook):
                        for i in range(len(acts)):
                            if positions[i] < activation.shape[1]:
                                activation[0, positions[i], :] = acts[i]
                        return activation
                    return hook_fn
                with torch.no_grad():
                    patched = model.run_with_hooks(corr_tokens,
                        fwd_hooks=extra + [(f'blocks.{layer}.hook_resid_post', make_patch(clean_acts, corr_qpos[:n_patch]))])
                patched_ld = compute_logit_diff(patched[0, -1], correct_tids, wrong_tids)
                recovery[mode][layer].append((patched_ld - corr_ld) / denom)

        if (t_idx+1) % 10 == 0:
            print(f'  {t_idx+1}/{TRIALS} done ({time.time()-t0:.0f}s)')
        flush_memory()

    print(f'\n  PI accuracy: normal={accuracy["normal"]/TRIALS:.0%}, ablated={accuracy["ablated"]/TRIALS:.0%}')
    print(f'\n  {"Layer":>6}  {"Normal":>10}  {"Ablated":>10}  {"Diff":>10}')
    print(f'  {"─" * 35}')
    for layer in test_layers:
        n_mean = np.mean(recovery['normal'][layer]) if recovery['normal'][layer] else 0
        a_mean = np.mean(recovery['ablated'][layer]) if recovery['ablated'][layer] else 0
        print(f'  L{layer:>4}  {n_mean:>+9.0%}  {a_mean:>+9.0%}  {a_mean-n_mean:>+9.0%}')

    save_step('appendix_a4_ablation_patching', {
        'config': {'keys': KEYS, 'updates': UPDATES, 'trials': TRIALS},
        'heads': [{'layer': l, 'head': h} for l, h in target_heads],
        'accuracy': {m: accuracy[m]/TRIALS for m in modes},
        'recovery': {m: {str(l): float(np.mean(recovery[m][l])) if recovery[m][l] else 0 for l in test_layers} for m in modes},
    })
    print(f'({time.time()-t0:.0f}s)')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# A5: Observational Head Metrics (exp 25c)
# 5 metrics from one forward pass: DLA, primacy, entropy, copy score, sensitivity
# Shows how observational methods compare to causal knockout (Step 5).
# ═══════════════════════════════════════════════════════════════════════════
TRIALS = CONFIG['main_trials']
print(f'A5: Observational Head Metrics at Point B ({KEYS}k, {UPDATES}u), {TRIALS} trials')
print('=' * 60)

# Accumulators per condition
obs = {}
for cond in ['RI', 'PI']:
    obs[cond] = {
        'dla': np.zeros((N_LAYERS, N_HEADS)),
        'primacy': np.zeros((N_LAYERS, N_HEADS)),
        'entropy': np.zeros((N_LAYERS, N_HEADS)),
        'copy_score': np.zeros((N_LAYERS, N_HEADS)),
        'retrieval': np.zeros((N_LAYERS, N_HEADS)),
        'n': 0,
    }
t0 = time.time()

for t_idx in range(TRIALS):
    for cond in ['RI', 'PI']:
        seed = hash((cond, t_idx, UPDATES, KEYS, 'a5')) % (2**31)
        trial = build_trial(KEYS, UPDATES, cond, seed)

        init_tid = VALUE_TO_TID.get(trial['initial_value'], -1)
        final_tid = VALUE_TO_TID.get(trial['final_value'], -1)
        init_tid_bare = tokenizer.encode(trial['initial_value'], add_special_tokens=False)[0]
        final_tid_bare = tokenizer.encode(trial['final_value'], add_special_tokens=False)[0]

        formatted = format_for_chat(trial['prompt'])
        tokens = model.to_tokens(formatted)
        token_ids = tokens[0].tolist()

        with torch.no_grad():
            logits, cache = model.run_with_cache(tokens, names_filter=lambda name: 'pattern' in name or 'hook_z' in name)

        pred_tid = logits[0, -1].argmax().item()
        init_pos = [i for i,t in enumerate(token_ids) if t == init_tid] if init_tid >= 0 else []
        final_pos = [i for i,t in enumerate(token_ids) if t == final_tid] if final_tid >= 0 else []

        logit_dir = model.W_U[:, init_tid_bare] - model.W_U[:, final_tid_bare]

        for layer in range(N_LAYERS):
            # DLA
            z = cache['z', layer][0, -1, :, :]
            W_O = model.W_O[layer]
            for head in range(N_HEADS):
                head_out = z[head] @ W_O[head]
                obs[cond]['dla'][layer, head] += (head_out @ logit_dir).item()

            # Attention
            pattern = cache['pattern', layer]
            attn = pattern[0, :, -1, :]
            for head in range(N_HEADS):
                h_attn = attn[head]
                a_init = h_attn[init_pos].sum().item() if init_pos else 0
                a_final = h_attn[final_pos].sum().item() if final_pos else 0
                obs[cond]['retrieval'][layer, head] += a_init + a_final
                denom = a_init + a_final
                obs[cond]['primacy'][layer, head] += (a_init / denom) if denom > 1e-8 else 0.5

                # Entropy
                p = h_attn.clamp(min=1e-10)
                obs[cond]['entropy'][layer, head] += -(p * p.log()).sum().item()

                # Copy score
                max_pos = h_attn.argmax().item()
                attended_tid = token_ids[max_pos] if max_pos < len(token_ids) else -1
                obs[cond]['copy_score'][layer, head] += float(attended_tid == pred_tid)

        obs[cond]['n'] += 1
        del cache

    if (t_idx+1) % 25 == 0:
        print(f'  {t_idx+1}/{TRIALS} done ({time.time()-t0:.0f}s)')
    flush_memory()

# Average
for cond in ['RI', 'PI']:
    n = obs[cond]['n']
    for key in ['dla', 'primacy', 'entropy', 'copy_score', 'retrieval']:
        obs[cond][key] /= n

# Condition sensitivity
cond_sensitivity = np.abs(obs['RI']['primacy'] - obs['PI']['primacy'])

# Rank correlation with Step 5 causal effect (if available)
print(f'\\nTop 5 heads by each PI metric:')
metrics = [
    ('DLA (→init)', obs['PI']['dla'], True),
    ('Primacy score', obs['PI']['primacy'], True),
    ('Retrieval', obs['PI']['retrieval'], True),
]
for name, scores, desc in metrics:
    flat = sorted([(scores[l,h],l,h) for l in range(N_LAYERS) for h in range(N_HEADS)], reverse=desc)
    heads_str = ', '.join(f'L{l}H{h}({s:+.2f})' for s,l,h in flat[:5])
    print(f'  {name}: {heads_str}')

save_step('appendix_a5_observational', {
    'config': {'keys': KEYS, 'updates': UPDATES, 'trials': TRIALS},
    'dla_pi': obs['PI']['dla'].tolist(),
    'primacy_pi': obs['PI']['primacy'].tolist(),
    'entropy_pi': obs['PI']['entropy'].tolist(),
    'copy_score_pi': obs['PI']['copy_score'].tolist(),
    'retrieval_pi': obs['PI']['retrieval'].tolist(),
    'condition_sensitivity': cond_sensitivity.tolist(),
})
print(f'({time.time()-t0:.0f}s)')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# A6: Logit Lens Across Operating Points (Step 2 at A, C, D)
# Shows how P(init) vs P(final) changes with interference level.
# Point B already done in Step 2. This adds the other points.
# ═══════════════════════════════════════════════════════════════════════════
TRIALS = CONFIG['main_trials']
extra_points = {k: v for k, v in CHAIN_STATE['operating_points'].items() if k != 'B' and v is not None}

if not extra_points:
    print('A6: SKIPPED — no additional operating points found')
else:
    print(f'A6: Logit Lens at {list(extra_points.keys())}, {TRIALS} trials each')
    print('=' * 60)

    all_point_results = {}
    t0 = time.time()

    for point_name, (pt_keys, pt_updates) in extra_points.items():
        print(f'\n  Point {point_name} ({pt_keys}k, {pt_updates}u):')

        total_needed = pt_keys * pt_updates
        if total_needed > len(VALUE_POOL):
            print(f'    SKIPPED — need {total_needed} values, have {len(VALUE_POOL)}')
            continue

        lp_init = {c: np.zeros(N_LAYERS) for c in ['RI', 'PI']}
        lp_final = {c: np.zeros(N_LAYERS) for c in ['RI', 'PI']}
        acc = {'RI': 0, 'PI': 0}

        for t_idx in range(TRIALS):
            for cond in ['RI', 'PI']:
                seed = hash((cond, t_idx, pt_updates, pt_keys, 'a6')) % (2**31)
                trial = build_trial(pt_keys, pt_updates, cond, seed)

                init_tids = get_tids(trial['initial_value'])
                final_tids = get_tids(trial['final_value'])

                formatted = format_for_chat(trial['prompt'])
                tokens = model.to_tokens(formatted)

                with torch.no_grad():
                    logits, cache = model.run_with_cache(tokens, names_filter=lambda name: 'resid_post' in name)

                pred = tokenizer.decode([logits[0, -1].argmax().item()]).strip()
                if pred.lower() == trial['expected'].lower():
                    acc[cond] += 1

                for layer in range(N_LAYERS):
                    resid = cache['resid_post', layer][0, -1, :]
                    ll = resid @ model.W_U + model.b_U
                    probs = torch.softmax(ll, dim=-1)
                    lp_init[cond][layer] += max(probs[t].item() for t in init_tids)
                    lp_final[cond][layer] += max(probs[t].item() for t in final_tids)

                del cache
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        for cond in ['RI', 'PI']:
            lp_init[cond] /= TRIALS
            lp_final[cond] /= TRIALS

        ri_acc = acc['RI'] / TRIALS
        pi_acc = acc['PI'] / TRIALS
        print(f'    Accuracy: RI={ri_acc:.0%}, PI={pi_acc:.0%}')
        print(f'    Last layer: RI P(init)={lp_init["RI"][-1]:.3f}, PI P(init)={lp_init["PI"][-1]:.3f}, PI P(final)={lp_final["PI"][-1]:.3f}')

        all_point_results[point_name] = {
            'keys': pt_keys, 'updates': pt_updates,
            'accuracy': {'RI': ri_acc, 'PI': pi_acc},
            'p_init_ri': lp_init['RI'].tolist(),
            'p_final_ri': lp_final['RI'].tolist(),
            'p_init_pi': lp_init['PI'].tolist(),
            'p_final_pi': lp_final['PI'].tolist(),
        }

        save_step('step2_logit_lens', {
            'config': {'keys': pt_keys, 'updates': pt_updates, 'trials': TRIALS},
            'accuracy': {'RI': ri_acc, 'PI': pi_acc},
            'p_init_ri': lp_init['RI'].tolist(),
            'p_final_ri': lp_final['RI'].tolist(),
            'p_init_pi': lp_init['PI'].tolist(),
            'p_final_pi': lp_final['PI'].tolist(),
        }, keys=pt_keys, updates=pt_updates)

    # Cross-point comparison
    print(f'\n{"=" * 60}')
    print('CROSS-POINT COMPARISON (last layer):')
    print(f'  {"Point":>6} {"Config":>8} {"RI acc":>8} {"PI acc":>8} {"RI P(i)":>8} {"PI P(i)":>8} {"PI P(f)":>8}')
    print(f'  {"─" * 55}')
    for pn, pr in all_point_results.items():
        cfg = f'{pr["keys"]}k,{pr["updates"]}u'
        print(f'  {pn:>6} {cfg:>8} {pr["accuracy"]["RI"]:>7.0%} {pr["accuracy"]["PI"]:>7.0%} '
              f'{pr["p_init_ri"][-1]:>7.3f} {pr["p_init_pi"][-1]:>7.3f} {pr["p_final_pi"][-1]:>7.3f}')

    print(f'\n({(time.time()-t0)/60:.1f} minutes)')